# All of Us ACE characterization: complete analysis pipeline

Run this notebook from top to bottom in an All of Us Controlled Tier workspace. Tables and figures are placed immediately after the analysis objects on which they depend. Only aggregate outputs are exported.


## Environment and output paths


In [ ]:
from pathlib import Path
import os
import sys


def locate_repository_root(start=Path.cwd()):
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "config.py").exists():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from within the aou-ace-characterization repository."
    )


REPO_ROOT = locate_repository_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.paths import ensure_output_directories

ensure_output_directories()
print(f"Repository root: {REPO_ROOT}")


from google.cloud import bigquery
from src.config import CDR_DATASET, PROJECT_ID

client = bigquery.Client(project=PROJECT_ID)
cdr = CDR_DATASET

print(f"Billing project: {PROJECT_ID}")
print(f"CDR dataset: {cdr}")


## Cohort eligibility and response completeness

Construct the EHHWB-eligible and respondent cohorts and export the aggregate cohort-flow counts.


In [ ]:
from google.cloud import bigquery

prerequisite_survey_ids = [
    1586134,  # The Basics
    1585710,  # Overall Health
    1585855,  # Lifestyle
]

ehhwb_survey_concept_id = 1703970

survey_counts = client.query(
    f"""
    WITH all_survey_people AS (
        SELECT DISTINCT
            person_id
        FROM `{cdr}.survey_conduct`
    ),

    prerequisite_completion AS (
        SELECT
            person_id,
            COUNT(DISTINCT survey_source_concept_id) AS n_prerequisites_completed
        FROM `{cdr}.survey_conduct`
        WHERE survey_source_concept_id IN UNNEST(@prerequisite_survey_ids)
        GROUP BY person_id
    ),

    eligible_people AS (
        SELECT
            person_id
        FROM prerequisite_completion
        WHERE n_prerequisites_completed = 3
    ),

    ehhwb_people AS (
        SELECT DISTINCT
            person_id
        FROM `{cdr}.survey_conduct`
        WHERE survey_source_concept_id = @ehhwb_survey_concept_id
    )

    SELECT
        (SELECT COUNT(*) FROM all_survey_people) AS n_unique_ids_in_survey_conduct,
        (SELECT COUNT(*) FROM eligible_people) AS n_completed_all_3_prerequisites,
        (SELECT COUNT(*) FROM ehhwb_people) AS n_completed_ehhwb
    """,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter(
                "prerequisite_survey_ids",
                "INT64",
                prerequisite_survey_ids,
            ),
            bigquery.ScalarQueryParameter(
                "ehhwb_survey_concept_id",
                "INT64",
                ehhwb_survey_concept_id,
            ),
        ]
    ),
).to_dataframe()

display(survey_counts)

In [ ]:
survey_counts.to_csv("outputs/figure_data/main/figure_1_flowchart.csv", index=False)
print("Saved: outputs/figure_data/main/figure_1_flowchart.csv")

## Main Table 1

Demographic comparison of EHHWB-eligible participants and EHHWB respondents.


In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np


# --------------------------------------------------
# Inputs
# --------------------------------------------------

prerequisite_survey_ids = [
    1586134,  # The Basics
    1585710,  # Overall Health
    1585855,  # Lifestyle
]

ehhwb_survey_concept_id = 1703970


# --------------------------------------------------
# Build eligible cohort, EHHWB indicator,
# and demographic variables
# --------------------------------------------------

table_1_raw = client.query(
    f"""
    WITH prerequisite_completion AS (
        SELECT
            person_id,
            COUNT(DISTINCT survey_source_concept_id)
                AS n_prerequisites_completed
        FROM `{cdr}.survey_conduct`
        WHERE survey_source_concept_id
            IN UNNEST(@prerequisite_survey_ids)
        GROUP BY person_id
    ),

    eligible_people AS (
        SELECT
            person_id
        FROM prerequisite_completion
        WHERE n_prerequisites_completed = 3
    ),

    earliest_ehhwb AS (
        SELECT
            person_id,
            survey_end_date
        FROM `{cdr}.survey_conduct`
        WHERE survey_source_concept_id
            = @ehhwb_survey_concept_id
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY person_id
            ORDER BY
                survey_end_date ASC,
                survey_conduct_id ASC
        ) = 1
    )

    SELECT
        e.person_id,

        CASE
            WHEN ehhwb.person_id IS NOT NULL THEN 1
            ELSE 0
        END AS completed_ehhwb,

        race.concept_name AS race,
        ethnicity.concept_name AS ethnicity,
        gender.concept_name AS gender,

        CASE
            WHEN p.year_of_birth IS NULL
                THEN 'Unknown'

            /*
            Use EHHWB completion date for respondents.
            For eligible nonrespondents, use the current year
            because they do not have an EHHWB survey date.
            */

            WHEN (
                COALESCE(
                    EXTRACT(YEAR FROM ehhwb.survey_end_date),
                    EXTRACT(YEAR FROM CURRENT_DATE())
                ) - p.year_of_birth
            ) BETWEEN 18 AND 30
                THEN '18-30'

            WHEN (
                COALESCE(
                    EXTRACT(YEAR FROM ehhwb.survey_end_date),
                    EXTRACT(YEAR FROM CURRENT_DATE())
                ) - p.year_of_birth
            ) BETWEEN 31 AND 45
                THEN '31-45'

            WHEN (
                COALESCE(
                    EXTRACT(YEAR FROM ehhwb.survey_end_date),
                    EXTRACT(YEAR FROM CURRENT_DATE())
                ) - p.year_of_birth
            ) BETWEEN 46 AND 60
                THEN '46-60'

            WHEN (
                COALESCE(
                    EXTRACT(YEAR FROM ehhwb.survey_end_date),
                    EXTRACT(YEAR FROM CURRENT_DATE())
                ) - p.year_of_birth
            ) BETWEEN 61 AND 75
                THEN '61-75'

            WHEN (
                COALESCE(
                    EXTRACT(YEAR FROM ehhwb.survey_end_date),
                    EXTRACT(YEAR FROM CURRENT_DATE())
                ) - p.year_of_birth
            ) >= 76
                THEN '76+'

            ELSE 'Unknown'
        END AS age_group

    FROM eligible_people e

    INNER JOIN `{cdr}.person` p
        ON e.person_id = p.person_id

    LEFT JOIN earliest_ehhwb ehhwb
        ON e.person_id = ehhwb.person_id

    LEFT JOIN `{cdr}.concept` race
        ON p.race_concept_id = race.concept_id

    LEFT JOIN `{cdr}.concept` ethnicity
        ON p.ethnicity_concept_id = ethnicity.concept_id

    LEFT JOIN `{cdr}.concept` gender
        ON p.gender_concept_id = gender.concept_id

    ORDER BY e.person_id
    """,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter(
                "prerequisite_survey_ids",
                "INT64",
                prerequisite_survey_ids,
            ),
            bigquery.ScalarQueryParameter(
                "ehhwb_survey_concept_id",
                "INT64",
                ehhwb_survey_concept_id,
            ),
        ]
    ),
).to_dataframe()


# --------------------------------------------------
# Cohort sanity checks
# --------------------------------------------------

assert table_1_raw["person_id"].is_unique, (
    "The eligible cohort contains duplicate person IDs."
)

assert table_1_raw["completed_ehhwb"].isin([0, 1]).all(), (
    "completed_ehhwb contains values other than 0 and 1."
)

n_eligible = table_1_raw["person_id"].nunique()

n_ehhwb = table_1_raw.loc[
    table_1_raw["completed_ehhwb"].eq(1),
    "person_id"
].nunique()

print(f"Eligible for EHHWB: {n_eligible:,}")
print(f"Eligible participants completing EHHWB: {n_ehhwb:,}")


# --------------------------------------------------
# Function to build one demographic section
# --------------------------------------------------

def build_table_1_section(
    df,
    group_col,
    characteristic_label,
    category_order=None
):
    temp = df.copy()

    # Make missing demographic values explicit
    temp[group_col] = (
        temp[group_col]
        .astype("object")
        .where(temp[group_col].notna(), "Missing")
    )

    eligible_counts = (
        temp
        .groupby(group_col, dropna=False)["person_id"]
        .nunique()
        .rename("Eligible for EHHWB n")
    )

    ehhwb_counts = (
        temp.loc[temp["completed_ehhwb"].eq(1)]
        .groupby(group_col, dropna=False)["person_id"]
        .nunique()
        .rename("EHHWB n")
    )

    section = (
        pd.concat(
            [eligible_counts, ehhwb_counts],
            axis=1
        )
        .fillna(0)
        .reset_index()
        .rename(columns={group_col: "Category"})
    )

    section["Eligible for EHHWB n"] = (
        section["Eligible for EHHWB n"].astype(int)
    )

    section["EHHWB n"] = (
        section["EHHWB n"].astype(int)
    )

    # Proportions use the full eligible and EHHWB denominators
    p_eligible = (
        section["Eligible for EHHWB n"] / n_eligible
    )

    p_ehhwb = (
        section["EHHWB n"] / n_ehhwb
    )

    section["Eligible for EHHWB, %"] = (
        p_eligible * 100
    )

    section["EHHWB, %"] = (
        p_ehhwb * 100
    )

    # Signed percentage-point difference:
    # EHHWB minus eligible
    section["Diff, %"] = (
        (p_ehhwb - p_eligible) * 100
    )

    # Absolute standardized mean difference
    pooled_sd = np.sqrt(
        (
            p_eligible * (1 - p_eligible)
            + p_ehhwb * (1 - p_ehhwb)
        ) / 2
    )

    section["Abs. Stand. Mean Diff."] = np.where(
        pooled_sd > 0,
        np.abs(p_ehhwb - p_eligible) / pooled_sd,
        0
    )

    section.insert(
        0,
        "Characteristic",
        characteristic_label
    )

    # Apply specified category ordering
    if category_order is not None:
        observed_categories = set(
            section["Category"].astype(str)
        )

        ordered_categories = [
            category
            for category in category_order
            if category in observed_categories
        ]

        additional_categories = [
            category
            for category in section["Category"].astype(str)
            if category not in ordered_categories
        ]

        final_order = (
            ordered_categories
            + additional_categories
        )

        section["Category"] = pd.Categorical(
            section["Category"].astype(str),
            categories=final_order,
            ordered=True
        )

        section = (
            section
            .sort_values("Category")
            .reset_index(drop=True)
        )

        section["Category"] = (
            section["Category"].astype(str)
        )

    else:
        section = (
            section
            .sort_values(
                "Eligible for EHHWB n",
                ascending=False
            )
            .reset_index(drop=True)
        )

    return section


# --------------------------------------------------
# Build demographic sections
# --------------------------------------------------

race_section = build_table_1_section(
    table_1_raw,
    group_col="race",
    characteristic_label="Race"
)

ethnicity_section = build_table_1_section(
    table_1_raw,
    group_col="ethnicity",
    characteristic_label="Ethnicity"
)

gender_section = build_table_1_section(
    table_1_raw,
    group_col="gender",
    characteristic_label="Gender"
)

age_section = build_table_1_section(
    table_1_raw,
    group_col="age_group",
    characteristic_label="Age Group",
    category_order=[
        "18-30",
        "31-45",
        "46-60",
        "61-75",
        "76+",
        "Unknown",
        "Missing"
    ]
)


# --------------------------------------------------
# Combine into Table 1
# --------------------------------------------------

table_1 = pd.concat(
    [
        race_section,
        ethnicity_section,
        gender_section,
        age_section
    ],
    ignore_index=True
)


# --------------------------------------------------
# Round and order columns
# --------------------------------------------------

table_1[
    [
        "Eligible for EHHWB, %",
        "EHHWB, %",
        "Diff, %",
        "Abs. Stand. Mean Diff."
    ]
] = table_1[
    [
        "Eligible for EHHWB, %",
        "EHHWB, %",
        "Diff, %",
        "Abs. Stand. Mean Diff."
    ]
].round(2)

table_1 = table_1[
    [
        "Characteristic",
        "Category",
        "Eligible for EHHWB n",
        "EHHWB n",
        "Eligible for EHHWB, %",
        "EHHWB, %",
        "Diff, %",
        "Abs. Stand. Mean Diff."
    ]
]


# --------------------------------------------------
# Validate section totals
# --------------------------------------------------

for characteristic in [
    "Race",
    "Ethnicity",
    "Gender",
    "Age Group"
]:
    eligible_section_total = table_1.loc[
        table_1["Characteristic"].eq(characteristic),
        "Eligible for EHHWB n"
    ].sum()

    ehhwb_section_total = table_1.loc[
        table_1["Characteristic"].eq(characteristic),
        "EHHWB n"
    ].sum()

    assert eligible_section_total == n_eligible, (
        f"{characteristic}: eligible counts do not "
        f"sum to {n_eligible:,}."
    )

    assert ehhwb_section_total == n_ehhwb, (
        f"{characteristic}: EHHWB counts do not "
        f"sum to {n_ehhwb:,}."
    )

    print(
        f"{characteristic}: "
        f"eligible = {eligible_section_total:,}; "
        f"EHHWB = {ehhwb_section_total:,}"
    )


display(table_1)

In [ ]:
# ==================================================
# SAVE OUTPUTS
# ==================================================

table_1.to_csv(
    "outputs/tables/main/table_1_demographic_comparison_of_ehhwb_eligible_participants_and_ehhwb_respondents.csv",
    index=False
)

print(
    "Saved: "
    "outputs/tables/main/table_1_demographic_comparison_of_ehhwb_eligible_participants_and_ehhwb_respondents.csv"
)

In [ ]:
"""
Check to see how many completed EHHWB survey and didn't complete required prerequisite surveys
"""

from google.cloud import bigquery

prerequisite_survey_ids = [
    1586134,  # The Basics
    1585710,  # Overall Health
    1585855,  # Lifestyle
]

ehhwb_survey_concept_id = 1703970

ehhwb_without_prereqs = client.query(
    f"""
    WITH prerequisite_completion AS (
        SELECT
            person_id,
            COUNT(DISTINCT survey_source_concept_id) AS n_prerequisites_completed
        FROM `{cdr}.survey_conduct`
        WHERE survey_source_concept_id IN UNNEST(@prerequisite_survey_ids)
        GROUP BY person_id
    ),

    eligible_people AS (
        SELECT
            person_id
        FROM prerequisite_completion
        WHERE n_prerequisites_completed = 3
    ),

    ehhwb_people AS (
        SELECT DISTINCT
            person_id
        FROM `{cdr}.survey_conduct`
        WHERE survey_source_concept_id = @ehhwb_survey_concept_id
    )

    SELECT
        COUNT(*) AS n_ehhwb_without_all_3_prerequisites
    FROM ehhwb_people e
    LEFT JOIN eligible_people p
        ON e.person_id = p.person_id
    WHERE p.person_id IS NULL
    """,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter(
                "prerequisite_survey_ids",
                "INT64",
                prerequisite_survey_ids,
            ),
            bigquery.ScalarQueryParameter(
                "ehhwb_survey_concept_id",
                "INT64",
                ehhwb_survey_concept_id,
            ),
        ]
    ),
).to_dataframe()

n_missing_prereqs = int(
    ehhwb_without_prereqs["n_ehhwb_without_all_3_prerequisites"].iloc[0]
)

print(
    f"EHHWB participants without all 3 prerequisite surveys: "
    f"{n_missing_prereqs:,}"
)

## Supplementary Table S1

Distribution of valid ACE questionnaire responses.


In [ ]:
from google.cloud import bigquery

ace_question_ids = [
    1332946, 1332947, 1332948, 1332950,
    1333208, 1332857, 1333210, 1333212,
    1332951, 1333202, 1333203
]

ehhwb_survey_concept_id = 1703970
skip_concept_id = 903096

s1_valid_response_distribution = client.query(
    f"""
    WITH earliest_ehhwb AS (
        SELECT
            person_id,
            survey_conduct_id,
            survey_start_datetime
        FROM `{cdr}.survey_conduct`
        WHERE survey_source_concept_id = @ehhwb_survey_concept_id
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY person_id
            ORDER BY survey_start_datetime ASC, survey_conduct_id ASC
        ) = 1
    ),

    questions AS (
        SELECT
            concept_id AS question_id
        FROM `{cdr}.concept`
        WHERE concept_id IN UNNEST(@ace_question_ids)
    ),

    raw_ace_responses AS (
        SELECT
            o.person_id,
            o.observation_source_concept_id AS question_id,
            o.value_as_concept_id,
            o.observation_datetime,
            o.observation_id
        FROM `{cdr}.observation` o
        INNER JOIN earliest_ehhwb e
            ON o.person_id = e.person_id
        WHERE o.observation_source_concept_id IN UNNEST(@ace_question_ids)
    ),

    deduped_ace_responses AS (
        SELECT
            person_id,
            question_id,
            value_as_concept_id
        FROM raw_ace_responses
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY person_id, question_id
            ORDER BY observation_datetime ASC, observation_id ASC
        ) = 1
    ),

    person_question_status AS (
        SELECT
            e.person_id,
            q.question_id,
            CASE
                WHEN a.person_id IS NULL THEN 'null_or_missing'
                WHEN a.value_as_concept_id IS NULL THEN 'null_or_missing'
                WHEN a.value_as_concept_id = @skip_concept_id THEN 'skip'
                ELSE 'value'
            END AS response_status
        FROM earliest_ehhwb e
        CROSS JOIN questions q
        LEFT JOIN deduped_ace_responses a
            ON e.person_id = a.person_id
           AND q.question_id = a.question_id
    ),

    person_counts AS (
        SELECT
            person_id,
            COUNTIF(response_status = 'value') AS n_valid_items
        FROM person_question_status
        GROUP BY person_id
    )

    SELECT
        n_valid_items AS number_of_valid_ace_responses,
        COUNT(*) AS participants_n,
        ROUND(
            100 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS participants_percent
    FROM person_counts
    GROUP BY n_valid_items
    ORDER BY n_valid_items
    """,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter(
                "ace_question_ids",
                "INT64",
                ace_question_ids
            ),
            bigquery.ScalarQueryParameter(
                "ehhwb_survey_concept_id",
                "INT64",
                ehhwb_survey_concept_id
            ),
            bigquery.ScalarQueryParameter(
                "skip_concept_id",
                "INT64",
                skip_concept_id
            ),
        ]
    ),
).to_dataframe()

display(s1_valid_response_distribution)



In [ ]:
s1_valid_response_distribution.to_csv(
    "outputs/tables/supplementary/supplementary_table_s1_valid_ace_response_distribution.csv",
    index=False
)

print("Saved: outputs/tables/supplementary/supplementary_table_s1_valid_ace_response_distribution.csv")

## Supplementary Table S2

ACE item response completeness.


In [ ]:
from google.cloud import bigquery

ace_question_ids = [
    1332946, 1332947, 1332948, 1332950,
    1333208, 1332857, 1333210, 1333212,
    1332951, 1333202, 1333203
]

ehhwb_survey_concept_id = 1703970
skip_concept_id = 903096

supplementary_table_s2 = client.query(
    f"""
    WITH earliest_ehhwb AS (
        SELECT
            person_id,
            survey_conduct_id,
            survey_start_datetime
        FROM `{cdr}.survey_conduct`
        WHERE survey_source_concept_id = @ehhwb_survey_concept_id
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY person_id
            ORDER BY survey_start_datetime ASC, survey_conduct_id ASC
        ) = 1
    ),

    questions AS (
        SELECT
            c.concept_id AS question_id,
            c.concept_name AS question_text
        FROM `{cdr}.concept` c
        WHERE c.concept_id IN UNNEST(@ace_question_ids)
    ),

    raw_ace_responses AS (
        SELECT
            o.person_id,
            o.observation_source_concept_id AS question_id,
            o.value_as_concept_id,
            o.observation_datetime,
            o.observation_id
        FROM `{cdr}.observation` o
        INNER JOIN earliest_ehhwb e
            ON o.person_id = e.person_id
        WHERE o.observation_source_concept_id IN UNNEST(@ace_question_ids)
    ),

    deduped_ace_responses AS (
        SELECT
            person_id,
            question_id,
            value_as_concept_id
        FROM raw_ace_responses
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY person_id, question_id
            ORDER BY observation_datetime ASC, observation_id ASC
        ) = 1
    ),

    person_question_status AS (
        SELECT
            e.person_id,
            q.question_id,
            q.question_text,
            CASE
                WHEN a.person_id IS NULL THEN 'missing_null'
                WHEN a.value_as_concept_id IS NULL THEN 'missing_null'
                WHEN a.value_as_concept_id = @skip_concept_id THEN 'skip'
                ELSE 'valid'
            END AS response_status
        FROM earliest_ehhwb e
        CROSS JOIN questions q
        LEFT JOIN deduped_ace_responses a
            ON e.person_id = a.person_id
           AND q.question_id = a.question_id
    )

    SELECT
        question_text,
        COUNT(*) AS participants_n,
        COUNTIF(response_status = 'valid') AS valid_responses_n,
        COUNTIF(response_status = 'skip') AS skipped_responses_n,
        COUNTIF(response_status = 'missing_null') AS missing_null_responses_n,
        ROUND(
            100 * COUNTIF(response_status = 'valid') / COUNT(*),
            2
        ) AS valid_responses_pct,
        ROUND(
            100 * COUNTIF(response_status = 'skip') / COUNT(*),
            2
        ) AS skipped_responses_pct,
        ROUND(
            100 * COUNTIF(response_status = 'missing_null') / COUNT(*),
            2
        ) AS missing_null_responses_pct
    FROM person_question_status
    GROUP BY
        question_id,
        question_text
    ORDER BY
        question_id
    """,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter(
                "ace_question_ids",
                "INT64",
                ace_question_ids
            ),
            bigquery.ScalarQueryParameter(
                "ehhwb_survey_concept_id",
                "INT64",
                ehhwb_survey_concept_id
            ),
            bigquery.ScalarQueryParameter(
                "skip_concept_id",
                "INT64",
                skip_concept_id
            ),
        ]
    ),
).to_dataframe()

supplementary_table_s2 = supplementary_table_s2.rename(
    columns={
        "question_text": "ACE Questionnaire Item",
        "participants_n": "Participants (N)",
        "valid_responses_n": "Valid Responses, n",
        "skipped_responses_n": "Skipped Responses, n",
        "missing_null_responses_n": "Missing (NULL) Responses, n",
        "valid_responses_pct": "Valid Responses, %",
        "skipped_responses_pct": "Skipped Responses, %",
        "missing_null_responses_pct": "Missing (NULL) Responses, %",
    }
)

display(supplementary_table_s2)

In [ ]:
supplementary_table_s2.to_csv(
    "outputs/tables/supplementary/supplementary_table_s2_ace_item_response_completeness.csv",
    index=False
)

print("Saved: outputs/tables/supplementary/supplementary_table_s2_ace_item_response_completeness.csv")

## Supplementary Table S3

ACE questionnaire completion by demographic subgroup.


In [ ]:
from google.cloud import bigquery
import pandas as pd


# --------------------------------------------------
# Inputs
# --------------------------------------------------

prerequisite_survey_ids = [
    1586134,  # The Basics
    1585710,  # Overall Health
    1585855,  # Lifestyle
]

ace_question_ids = [
    1332946, 1332947, 1332948, 1332950,
    1333208, 1332857, 1333210, 1333212,
    1332951, 1333202, 1333203
]

ehhwb_survey_concept_id = 1703970
skip_concept_id = 903096


# --------------------------------------------------
# Rebuild person-level ACE questionnaire completion
# among EHHWB respondents who completed all three
# prerequisite surveys
# --------------------------------------------------

person_level_completion = client.query(
    f"""
    WITH prerequisite_completion AS (
        SELECT
            person_id,
            COUNT(DISTINCT survey_source_concept_id)
                AS n_prerequisites_completed
        FROM `{cdr}.survey_conduct`
        WHERE survey_source_concept_id
            IN UNNEST(@prerequisite_survey_ids)
        GROUP BY person_id
    ),

    eligible_people AS (
        SELECT
            person_id
        FROM prerequisite_completion
        WHERE n_prerequisites_completed = 3
    ),

    earliest_ehhwb AS (
        SELECT
            sc.person_id,
            sc.survey_conduct_id,
            sc.survey_end_date
        FROM `{cdr}.survey_conduct` sc

        INNER JOIN eligible_people e
            ON sc.person_id = e.person_id

        WHERE sc.survey_source_concept_id
            = @ehhwb_survey_concept_id

        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY sc.person_id
            ORDER BY
                sc.survey_end_date ASC,
                sc.survey_conduct_id ASC
        ) = 1
    ),

    questions AS (
        SELECT
            question_id
        FROM UNNEST(@ace_question_ids) AS question_id
    ),

    raw_ace_responses AS (
        SELECT
            o.person_id,
            o.observation_source_concept_id AS question_id,
            o.value_as_concept_id,
            o.observation_datetime,
            o.observation_id
        FROM `{cdr}.observation` o

        INNER JOIN earliest_ehhwb e
            ON o.person_id = e.person_id

        WHERE o.observation_source_concept_id
            IN UNNEST(@ace_question_ids)
    ),

    deduped_ace_responses AS (
        SELECT
            person_id,
            question_id,
            value_as_concept_id
        FROM raw_ace_responses

        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY person_id, question_id
            ORDER BY
                observation_datetime ASC,
                observation_id ASC
        ) = 1
    ),

    person_question_status AS (
        SELECT
            e.person_id,
            q.question_id,

            CASE
                WHEN r.person_id IS NULL
                    THEN 'missing_null'

                WHEN r.value_as_concept_id IS NULL
                    THEN 'missing_null'

                WHEN r.value_as_concept_id = @skip_concept_id
                    THEN 'skip'

                ELSE 'valid'
            END AS response_status

        FROM earliest_ehhwb e

        CROSS JOIN questions q

        LEFT JOIN deduped_ace_responses r
            ON e.person_id = r.person_id
           AND q.question_id = r.question_id
    ),

    person_completion AS (
        SELECT
            person_id,
            COUNTIF(response_status = 'valid')
                AS n_valid,
            COUNTIF(response_status = 'skip')
                AS n_skipped,
            COUNTIF(response_status = 'missing_null')
                AS n_missing_null
        FROM person_question_status
        GROUP BY person_id
    ),

    demographics AS (
        SELECT
            e.person_id,

            gender.concept_name AS gender,
            race.concept_name AS race,
            ethnicity.concept_name AS ethnicity,

            EXTRACT(YEAR FROM e.survey_end_date)
                - p.year_of_birth AS age_at_ehhwb,

            CASE
                WHEN p.year_of_birth IS NULL
                    OR e.survey_end_date IS NULL
                    THEN 'Unknown'

                WHEN EXTRACT(YEAR FROM e.survey_end_date)
                    - p.year_of_birth BETWEEN 18 AND 30
                    THEN '18-30'

                WHEN EXTRACT(YEAR FROM e.survey_end_date)
                    - p.year_of_birth BETWEEN 31 AND 45
                    THEN '31-45'

                WHEN EXTRACT(YEAR FROM e.survey_end_date)
                    - p.year_of_birth BETWEEN 46 AND 60
                    THEN '46-60'

                WHEN EXTRACT(YEAR FROM e.survey_end_date)
                    - p.year_of_birth BETWEEN 61 AND 75
                    THEN '61-75'

                WHEN EXTRACT(YEAR FROM e.survey_end_date)
                    - p.year_of_birth >= 76
                    THEN '76+'

                ELSE 'Unknown'
            END AS age_group

        FROM earliest_ehhwb e

        INNER JOIN `{cdr}.person` p
            ON e.person_id = p.person_id

        LEFT JOIN `{cdr}.concept` gender
            ON p.gender_concept_id = gender.concept_id

        LEFT JOIN `{cdr}.concept` race
            ON p.race_concept_id = race.concept_id

        LEFT JOIN `{cdr}.concept` ethnicity
            ON p.ethnicity_concept_id = ethnicity.concept_id
    )

    SELECT
        pc.person_id,
        pc.n_valid,
        pc.n_skipped,
        pc.n_missing_null,
        d.race,
        d.ethnicity,
        d.gender,
        d.age_at_ehhwb,
        d.age_group

    FROM person_completion pc

    LEFT JOIN demographics d
        ON pc.person_id = d.person_id

    ORDER BY pc.person_id
    """,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter(
                "prerequisite_survey_ids",
                "INT64",
                prerequisite_survey_ids
            ),
            bigquery.ArrayQueryParameter(
                "ace_question_ids",
                "INT64",
                ace_question_ids
            ),
            bigquery.ScalarQueryParameter(
                "ehhwb_survey_concept_id",
                "INT64",
                ehhwb_survey_concept_id
            ),
            bigquery.ScalarQueryParameter(
                "skip_concept_id",
                "INT64",
                skip_concept_id
            ),
        ]
    ),
).to_dataframe()


# --------------------------------------------------
# Sanity checks
# --------------------------------------------------

assert person_level_completion["person_id"].is_unique, (
    "person_level_completion contains duplicate person IDs."
)

item_totals = person_level_completion[
    ["n_valid", "n_skipped", "n_missing_null"]
].sum(axis=1)

assert item_totals.eq(11).all(), (
    "At least one participant does not sum to 11 ACE items."
)

n_ehhwb = person_level_completion["person_id"].nunique()

n_complete_11 = (
    person_level_completion["n_valid"]
    .eq(11)
    .sum()
)

print(
    "Prerequisite-eligible participants completing EHHWB: "
    f"{n_ehhwb:,}"
)

print(
    "Participants completing all 11 ACE items: "
    f"{n_complete_11:,}"
)


# Optional exact comparison with Table 1 if it is
# already present in the notebook
if "table_1_raw" in globals():
    table_1_ehhwb_ids = set(
        table_1_raw.loc[
            table_1_raw["completed_ehhwb"].eq(1),
            "person_id"
        ]
    )

    s3_ehhwb_ids = set(
        person_level_completion["person_id"]
    )

    assert s3_ehhwb_ids == table_1_ehhwb_ids, (
        "S3 and Table 1 do not contain the same "
        "EHHWB respondent cohort."
    )

    print(
        "Confirmed: S3 uses exactly the same "
        "EHHWB respondent cohort as Table 1."
    )


# --------------------------------------------------
# Summarization function
# --------------------------------------------------

def build_completion_section(
    df,
    group_col,
    characteristic_label,
    category_order=None
):
    temp = df.copy()

    # Make missing demographic values explicit
    temp[group_col] = (
        temp[group_col]
        .astype("object")
        .where(
            temp[group_col].notna(),
            "Missing"
        )
    )

    section = (
        temp
        .groupby(group_col, dropna=False)
        .agg(
            n=("person_id", "nunique"),

            mean_valid_responses=(
                "n_valid",
                "mean"
            ),

            mean_skipped_responses=(
                "n_skipped",
                "mean"
            ),

            mean_missing_null_responses=(
                "n_missing_null",
                "mean"
            ),

            completed_all_11_items_pct=(
                "n_valid",
                lambda x: 100 * x.eq(11).mean()
            )
        )
        .reset_index()
        .rename(columns={group_col: "Name"})
    )

    section.insert(
        0,
        "Characteristic",
        characteristic_label
    )

    if category_order is not None:
        observed_categories = set(
            section["Name"].astype(str)
        )

        ordered_categories = [
            category
            for category in category_order
            if category in observed_categories
        ]

        additional_categories = [
            category
            for category in section["Name"].astype(str)
            if category not in ordered_categories
        ]

        final_order = (
            ordered_categories
            + additional_categories
        )

        section["Name"] = pd.Categorical(
            section["Name"].astype(str),
            categories=final_order,
            ordered=True
        )

        section = (
            section
            .sort_values("Name")
            .reset_index(drop=True)
        )

        section["Name"] = (
            section["Name"]
            .astype(str)
        )

    else:
        section = (
            section
            .sort_values(
                "n",
                ascending=False
            )
            .reset_index(drop=True)
        )

    return section


# --------------------------------------------------
# Build all four demographic sections
# --------------------------------------------------

race_section = build_completion_section(
    person_level_completion,
    group_col="race",
    characteristic_label="Race"
)

ethnicity_section = build_completion_section(
    person_level_completion,
    group_col="ethnicity",
    characteristic_label="Ethnicity"
)

gender_section = build_completion_section(
    person_level_completion,
    group_col="gender",
    characteristic_label="Gender"
)

age_section = build_completion_section(
    person_level_completion,
    group_col="age_group",
    characteristic_label="Age Group",
    category_order=[
        "18-30",
        "31-45",
        "46-60",
        "61-75",
        "76+",
        "Unknown",
        "Missing"
    ]
)


# --------------------------------------------------
# Combine into Supplementary Table S3
# --------------------------------------------------

supplementary_table_s3 = pd.concat(
    [
        race_section,
        ethnicity_section,
        gender_section,
        age_section
    ],
    ignore_index=True
)


# --------------------------------------------------
# Round and rename columns
# --------------------------------------------------

columns_to_round = [
    "mean_valid_responses",
    "mean_skipped_responses",
    "mean_missing_null_responses",
    "completed_all_11_items_pct"
]

supplementary_table_s3[columns_to_round] = (
    supplementary_table_s3[
        columns_to_round
    ].round(2)
)

supplementary_table_s3 = (
    supplementary_table_s3.rename(
        columns={
            "mean_valid_responses":
                "Mean Valid Responses",

            "mean_skipped_responses":
                "Mean Skipped Responses",

            "mean_missing_null_responses":
                "Mean Missing (NULL) Response",

            "completed_all_11_items_pct":
                "Completed All 11 Items, %"
        }
    )
)

supplementary_table_s3 = supplementary_table_s3[
    [
        "Characteristic",
        "Name",
        "n",
        "Mean Valid Responses",
        "Mean Skipped Responses",
        "Mean Missing (NULL) Response",
        "Completed All 11 Items, %"
    ]
]


# --------------------------------------------------
# Validate that each demographic section sums to
# the prerequisite-eligible EHHWB cohort
# --------------------------------------------------

for characteristic in [
    "Race",
    "Ethnicity",
    "Gender",
    "Age Group"
]:
    section_total = supplementary_table_s3.loc[
        supplementary_table_s3[
            "Characteristic"
        ].eq(characteristic),
        "n"
    ].sum()

    assert section_total == n_ehhwb, (
        f"{characteristic}: section total "
        f"{int(section_total):,} does not equal "
        f"the S3 cohort total of {n_ehhwb:,}."
    )

    print(
        f"{characteristic} section total: "
        f"{int(section_total):,}"
    )


# --------------------------------------------------
# Display Supplementary Table S3
# --------------------------------------------------

display(supplementary_table_s3)

In [ ]:
output_filename = (
    "outputs/tables/supplementary/supplementary_table_s3_ace_questionnaire_"
    "completion_by_demographic_subgroup.csv"
)

supplementary_table_s3.to_csv(
    output_filename,
    index=False
)

print(f"Saved: {output_filename}")


## ACE scoring and Main Table 2

Construct the eight-domain ACE burden score and summarize its distribution.


In [ ]:
from google.cloud import bigquery
import pandas as pd

# ==================================================
# QUESTION CONCEPT IDS
# ==================================================

# Single-item ACE domains
mental_illness_question_id = 1332946
incarceration_question_id = 1332950
parental_separation_question_id = 1333208
witnessed_violence_question_id = 1332857
physical_abuse_question_id = 1333210
emotional_abuse_question_id = 1333212

# Two questions combined into one household substance-use domain
alcohol_question_id = 1332947
drug_question_id = 1332948

# Three questions combined into one sexual-abuse domain
sexual_touch_question_id = 1332951
sexual_touch_them_question_id = 1333202
forced_sex_question_id = 1333203

ace_question_ids = [
    mental_illness_question_id,
    alcohol_question_id,
    drug_question_id,
    incarceration_question_id,
    parental_separation_question_id,
    witnessed_violence_question_id,
    physical_abuse_question_id,
    emotional_abuse_question_id,
    sexual_touch_question_id,
    sexual_touch_them_question_id,
    forced_sex_question_id,
]


# ==================================================
# ANSWER CONCEPT IDS
# ==================================================

yes_concept_id = 1704072          # Yes
no_concept_id = 1704093           # No

once_concept_id = 1704061         # Once
never_concept_id = 1704063        # Never
more_than_once_concept_id = 1704088  # More than once

parents_not_married_concept_id = 1704106  # Parents not married
skip_concept_id = 903096                 # PMI: Skip

ehhwb_survey_concept_id = 1703970        # Emotional Health History and Well-Being


# ==================================================
# BUILD PERSON-LEVEL 8-DOMAIN ACE SCORES
# ==================================================

ace_8_domain_person_level = client.query(
    f"""
    WITH earliest_ehhwb AS (
        SELECT
            person_id,
            survey_conduct_id,
            survey_end_date
        FROM `{cdr}.survey_conduct`
        WHERE survey_source_concept_id = @ehhwb_survey_concept_id
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY person_id
            ORDER BY
                survey_end_date ASC,
                survey_conduct_id ASC
        ) = 1
    ),

    raw_ace_responses AS (
        SELECT
            o.person_id,
            o.observation_source_concept_id AS question_id,
            o.value_as_concept_id AS answer_id,
            o.observation_datetime,
            o.observation_id
        FROM `{cdr}.observation` o
        INNER JOIN earliest_ehhwb e
            ON o.person_id = e.person_id
        WHERE o.observation_source_concept_id
            IN UNNEST(@ace_question_ids)
    ),

    deduped_ace_responses AS (
        SELECT
            person_id,
            question_id,
            answer_id
        FROM raw_ace_responses
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY person_id, question_id
            ORDER BY
                observation_datetime ASC,
                observation_id ASC
        ) = 1
    ),

    all_person_questions AS (
        SELECT
            e.person_id,
            question_id
        FROM earliest_ehhwb e
        CROSS JOIN UNNEST(@ace_question_ids) AS question_id
    ),

    person_question_answers AS (
        SELECT
            pq.person_id,
            pq.question_id,
            r.answer_id,

            CASE
                -- Yes = ACE present
                WHEN r.answer_id = @yes_concept_id THEN 1

                -- No = ACE absent
                WHEN r.answer_id = @no_concept_id THEN 0

                -- Once = ACE present
                WHEN r.answer_id = @once_concept_id THEN 1

                -- More than once = ACE present
                WHEN r.answer_id = @more_than_once_concept_id THEN 1

                -- Never = ACE absent
                WHEN r.answer_id = @never_concept_id THEN 0

                -- Parents not married = indeterminate/missing
                WHEN r.answer_id = @parents_not_married_concept_id THEN NULL

                -- PMI: Skip = indeterminate/missing
                WHEN r.answer_id = @skip_concept_id THEN NULL

                -- Includes NULL, Don't know/Not sure,
                -- Prefer not to answer, and any unexpected response
                ELSE NULL
            END AS item_score

        FROM all_person_questions pq
        LEFT JOIN deduped_ace_responses r
            ON pq.person_id = r.person_id
           AND pq.question_id = r.question_id
    ),

    person_items_wide AS (
        SELECT
            person_id,

            -- Household mental illness
            MAX(IF(
                question_id = @mental_illness_question_id,
                item_score,
                NULL
            )) AS mental_illness_item,

            -- Household alcohol misuse
            MAX(IF(
                question_id = @alcohol_question_id,
                item_score,
                NULL
            )) AS alcohol_item,

            -- Household drug/prescription medication misuse
            MAX(IF(
                question_id = @drug_question_id,
                item_score,
                NULL
            )) AS drug_item,

            -- Incarcerated household member
            MAX(IF(
                question_id = @incarceration_question_id,
                item_score,
                NULL
            )) AS incarceration_item,

            -- Parental separation or divorce
            MAX(IF(
                question_id = @parental_separation_question_id,
                item_score,
                NULL
            )) AS parental_separation_item,

            -- Witnessed violence between adults in the home
            MAX(IF(
                question_id = @witnessed_violence_question_id,
                item_score,
                NULL
            )) AS witnessed_violence_item,

            -- Physical abuse
            MAX(IF(
                question_id = @physical_abuse_question_id,
                item_score,
                NULL
            )) AS physical_abuse_item,

            -- Emotional abuse
            MAX(IF(
                question_id = @emotional_abuse_question_id,
                item_score,
                NULL
            )) AS emotional_abuse_item,

            -- Sexual touching
            MAX(IF(
                question_id = @sexual_touch_question_id,
                item_score,
                NULL
            )) AS sexual_touch_item,

            -- Made participant touch another person sexually
            MAX(IF(
                question_id = @sexual_touch_them_question_id,
                item_score,
                NULL
            )) AS sexual_touch_them_item,

            -- Forced sex
            MAX(IF(
                question_id = @forced_sex_question_id,
                item_score,
                NULL
            )) AS forced_sex_item

        FROM person_question_answers
        GROUP BY person_id
    ),

    eight_domains AS (
        SELECT
            person_id,

            -- Domain 1: household mental illness
            mental_illness_item AS household_mental_illness,

            -- Domain 2: household substance use
            CASE
                -- Yes to either alcohol or drugs = domain present
                WHEN alcohol_item = 1 OR drug_item = 1 THEN 1

                -- No to both = domain absent
                WHEN alcohol_item = 0 AND drug_item = 0 THEN 0

                -- No affirmative response, but one or both are missing
                ELSE NULL
            END AS household_substance_use,

            -- Domain 3: incarcerated household member
            incarceration_item AS incarcerated_household_member,

            -- Domain 4: parental separation/divorce
            parental_separation_item AS parental_separation_divorce,

            -- Domain 5: witnessed household violence
            witnessed_violence_item AS witnessed_household_violence,

            -- Domain 6: physical abuse
            physical_abuse_item AS physical_abuse,

            -- Domain 7: emotional abuse
            emotional_abuse_item AS emotional_abuse,

            -- Domain 8: sexual abuse
            CASE
                -- Positive response to any of the 3 questions
                WHEN sexual_touch_item = 1
                  OR sexual_touch_them_item = 1
                  OR forced_sex_item = 1
                    THEN 1

                -- All 3 explicitly answered Never
                WHEN sexual_touch_item = 0
                  AND sexual_touch_them_item = 0
                  AND forced_sex_item = 0
                    THEN 0

                -- No positive response, but at least one item is missing
                ELSE NULL
            END AS sexual_abuse

        FROM person_items_wide
    ),

    scored_people AS (
        SELECT
            *,

            (
                CAST(household_mental_illness IS NOT NULL AS INT64)
                + CAST(household_substance_use IS NOT NULL AS INT64)
                + CAST(incarcerated_household_member IS NOT NULL AS INT64)
                + CAST(parental_separation_divorce IS NOT NULL AS INT64)
                + CAST(witnessed_household_violence IS NOT NULL AS INT64)
                + CAST(physical_abuse IS NOT NULL AS INT64)
                + CAST(emotional_abuse IS NOT NULL AS INT64)
                + CAST(sexual_abuse IS NOT NULL AS INT64)
            ) AS n_classifiable_domains,

            CASE
                WHEN household_mental_illness IS NOT NULL
                 AND household_substance_use IS NOT NULL
                 AND incarcerated_household_member IS NOT NULL
                 AND parental_separation_divorce IS NOT NULL
                 AND witnessed_household_violence IS NOT NULL
                 AND physical_abuse IS NOT NULL
                 AND emotional_abuse IS NOT NULL
                 AND sexual_abuse IS NOT NULL
                THEN
                    household_mental_illness
                    + household_substance_use
                    + incarcerated_household_member
                    + parental_separation_divorce
                    + witnessed_household_violence
                    + physical_abuse
                    + emotional_abuse
                    + sexual_abuse
                ELSE NULL
            END AS ace_burden_score

        FROM eight_domains
    )

    SELECT *
    FROM scored_people
    ORDER BY person_id
    """,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter(
                "ace_question_ids",
                "INT64",
                ace_question_ids
            ),

            bigquery.ScalarQueryParameter(
                "ehhwb_survey_concept_id",
                "INT64",
                ehhwb_survey_concept_id
            ),

            # Question IDs
            bigquery.ScalarQueryParameter(
                "mental_illness_question_id",
                "INT64",
                mental_illness_question_id
            ),
            bigquery.ScalarQueryParameter(
                "alcohol_question_id",
                "INT64",
                alcohol_question_id
            ),
            bigquery.ScalarQueryParameter(
                "drug_question_id",
                "INT64",
                drug_question_id
            ),
            bigquery.ScalarQueryParameter(
                "incarceration_question_id",
                "INT64",
                incarceration_question_id
            ),
            bigquery.ScalarQueryParameter(
                "parental_separation_question_id",
                "INT64",
                parental_separation_question_id
            ),
            bigquery.ScalarQueryParameter(
                "witnessed_violence_question_id",
                "INT64",
                witnessed_violence_question_id
            ),
            bigquery.ScalarQueryParameter(
                "physical_abuse_question_id",
                "INT64",
                physical_abuse_question_id
            ),
            bigquery.ScalarQueryParameter(
                "emotional_abuse_question_id",
                "INT64",
                emotional_abuse_question_id
            ),
            bigquery.ScalarQueryParameter(
                "sexual_touch_question_id",
                "INT64",
                sexual_touch_question_id
            ),
            bigquery.ScalarQueryParameter(
                "sexual_touch_them_question_id",
                "INT64",
                sexual_touch_them_question_id
            ),
            bigquery.ScalarQueryParameter(
                "forced_sex_question_id",
                "INT64",
                forced_sex_question_id
            ),

            # Answer IDs
            bigquery.ScalarQueryParameter(
                "yes_concept_id",
                "INT64",
                yes_concept_id
            ),
            bigquery.ScalarQueryParameter(
                "no_concept_id",
                "INT64",
                no_concept_id
            ),
            bigquery.ScalarQueryParameter(
                "once_concept_id",
                "INT64",
                once_concept_id
            ),
            bigquery.ScalarQueryParameter(
                "never_concept_id",
                "INT64",
                never_concept_id
            ),
            bigquery.ScalarQueryParameter(
                "more_than_once_concept_id",
                "INT64",
                more_than_once_concept_id
            ),
            bigquery.ScalarQueryParameter(
                "parents_not_married_concept_id",
                "INT64",
                parents_not_married_concept_id
            ),
            bigquery.ScalarQueryParameter(
                "skip_concept_id",
                "INT64",
                skip_concept_id
            ),
        ]
    ),
).to_dataframe()


# ==================================================
# PERSON-LEVEL SANITY CHECKS
# ==================================================

display(ace_8_domain_person_level.head())

print(f"EHHWB participants: {len(ace_8_domain_person_level):,}")

print(
    "Participants with all 8 ACE domains classifiable: "
    f"{ace_8_domain_person_level['ace_burden_score'].notna().sum():,}"
)

print(
    "Participants with at least 1 indeterminate ACE domain: "
    f"{ace_8_domain_person_level['ace_burden_score'].isna().sum():,}"
)

display(
    ace_8_domain_person_level[
        "n_classifiable_domains"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("Number of classifiable ACE domains")
    .reset_index(name="Participants, n")
)


# ==================================================
# TABLE 2: DISTRIBUTION OF 0-8 ACE BURDEN SCORES
# ==================================================

complete_ace_scores = (
    ace_8_domain_person_level
    .loc[
        ace_8_domain_person_level["ace_burden_score"].notna(),
        ["person_id", "ace_burden_score"]
    ]
    .copy()
)

complete_ace_scores["ace_burden_score"] = (
    complete_ace_scores["ace_burden_score"].astype(int)
)

score_counts = (
    complete_ace_scores["ace_burden_score"]
    .value_counts()
    .reindex(range(0, 9), fill_value=0)
    .sort_index()
)

table_2_ace_distribution = pd.DataFrame({
    "ACE Burden Score": score_counts.index,
    "n": score_counts.values
})

total_complete = int(table_2_ace_distribution["n"].sum())

table_2_ace_distribution["%"] = (
    100 * table_2_ace_distribution["n"] / total_complete
)

# Reverse cumulative sum gives number with score >= current score
table_2_ace_distribution["Cumulative n (≥ score)"] = (
    table_2_ace_distribution["n"]
    .iloc[::-1]
    .cumsum()
    .iloc[::-1]
)

table_2_ace_distribution["Cumulative % (≥ score)"] = (
    100
    * table_2_ace_distribution["Cumulative n (≥ score)"]
    / total_complete
)

table_2_ace_distribution[
    ["%", "Cumulative % (≥ score)"]
] = table_2_ace_distribution[
    ["%", "Cumulative % (≥ score)"]
].round(2)

display(table_2_ace_distribution)



# ==================================================
# FINAL VALIDATION
# ==================================================

assert table_2_ace_distribution["ACE Burden Score"].tolist() == list(range(9))

assert table_2_ace_distribution["n"].sum() == (
    ace_8_domain_person_level["ace_burden_score"].notna().sum()
)

assert table_2_ace_distribution.loc[
    table_2_ace_distribution["ACE Burden Score"] == 0,
    "Cumulative n (≥ score)"
].iloc[0] == total_complete

assert table_2_ace_distribution.loc[
    table_2_ace_distribution["ACE Burden Score"] == 8,
    "Cumulative n (≥ score)"
].iloc[0] == table_2_ace_distribution.loc[
    table_2_ace_distribution["ACE Burden Score"] == 8,
    "n"
].iloc[0]

print("All validation checks passed.")

In [ ]:
# ==================================================
# SAVE OUTPUTS
# ==================================================

table_2_ace_distribution.to_csv(
    "outputs/tables/main/table_2_distribution_of_8_domain_ace_burden_scores.csv",
    index=False
)


print(
    "Saved: outputs/tables/main/table_2_distribution_of_8_domain_ace_burden_scores.csv"
)

## Main Figure 1

Cohort flow and demographic representativeness.

**Required inputs:** cohort-flow data, Main Table 1, Supplementary Tables S1 and S3, and Main Table 2. Run those sections first if their exported CSV files are not present.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


# ==================================================
# INPUT FILES
# ==================================================

flowchart_file = "outputs/figure_data/main/figure_1_flowchart.csv"

demographic_file = (
    "outputs/tables/main/table_1_demographic_comparison_of_ehhwb_eligible_participants_"
    "and_ehhwb_respondents.csv"
)

valid_response_distribution_file = (
    "outputs/tables/supplementary/supplementary_table_s1_valid_ace_response_distribution.csv"
)

completion_by_demographic_file = (
    "outputs/tables/supplementary/supplementary_table_s3_ace_questionnaire_completion_"
    "by_demographic_subgroup.csv"
)

eight_domain_distribution_file = (
    "outputs/tables/main/table_2_distribution_of_8_domain_ace_burden_scores.csv"
)


# ==================================================
# LOAD INPUT FILES
# ==================================================

flowchart_df = pd.read_csv(
    flowchart_file
)

demographic_df = pd.read_csv(
    demographic_file
)

valid_response_distribution_df = pd.read_csv(
    valid_response_distribution_file
)

completion_by_demographic_df = pd.read_csv(
    completion_by_demographic_file
)

eight_domain_distribution_df = pd.read_csv(
    eight_domain_distribution_file
)


# ==================================================
# VALIDATE INPUT FILES
# ==================================================

required_flowchart_columns = [
    "n_unique_ids_in_survey_conduct",
    "n_completed_all_3_prerequisites",
    "n_completed_ehhwb"
]

required_demographic_columns = [
    "Characteristic",
    "Category",
    "Eligible for EHHWB n",
    "EHHWB n",
    "Eligible for EHHWB, %",
    "EHHWB, %",
    "Diff, %",
    "Abs. Stand. Mean Diff."
]

required_valid_distribution_columns = [
    "number_of_valid_ace_responses",
    "participants_n",
    "participants_percent"
]

required_completion_columns = [
    "Characteristic",
    "Name",
    "n",
    "Mean Valid Responses",
    "Mean Skipped Responses",
    "Mean Missing (NULL) Response",
    "Completed All 11 Items, %"
]

required_eight_domain_columns = [
    "ACE Burden Score",
    "n",
    "%",
    "Cumulative n (≥ score)",
    "Cumulative % (≥ score)"
]


def validate_columns(
    dataframe,
    required_columns,
    dataframe_name
):
    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    assert not missing_columns, (
        f"{dataframe_name} is missing required columns: "
        + ", ".join(missing_columns)
    )


validate_columns(
    flowchart_df,
    required_flowchart_columns,
    "Flowchart CSV"
)

validate_columns(
    demographic_df,
    required_demographic_columns,
    "Table 1 demographic CSV"
)

validate_columns(
    valid_response_distribution_df,
    required_valid_distribution_columns,
    "Supplementary Table S1 CSV"
)

validate_columns(
    completion_by_demographic_df,
    required_completion_columns,
    "Supplementary Table S3 CSV"
)

validate_columns(
    eight_domain_distribution_df,
    required_eight_domain_columns,
    "Table 2 eight-domain CSV"
)


assert len(flowchart_df) == 1, (
    "The flowchart CSV must contain exactly one row."
)

assert not demographic_df.empty, (
    "The Table 1 demographic CSV is empty."
)

assert not valid_response_distribution_df.empty, (
    "The Supplementary Table S1 CSV is empty."
)

assert not completion_by_demographic_df.empty, (
    "The Supplementary Table S3 CSV is empty."
)

assert not eight_domain_distribution_df.empty, (
    "The Table 2 eight-domain CSV is empty."
)


# ==================================================
# SET FONTS
# ==================================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [
        "Helvetica",
        "Arial",
        "Liberation Sans",
        "DejaVu Sans"
    ],
    "mathtext.fontset": "dejavusans",
    "axes.unicode_minus": False
})


# ==================================================
# DERIVE COHORT COUNTS
# ==================================================

flow_row = flowchart_df.iloc[0]

n_survey_conduct = int(
    flow_row[
        "n_unique_ids_in_survey_conduct"
    ]
)

n_completed_prerequisites = int(
    flow_row[
        "n_completed_all_3_prerequisites"
    ]
)

# This is the total number of EHHWB respondents,
# regardless of prerequisite completion.
n_all_ehhwb = int(
    flow_row[
        "n_completed_ehhwb"
    ]
)


# --------------------------------------------------
# Derive prerequisite-eligible EHHWB respondents
# from Table 1
# --------------------------------------------------

table_1_section_totals = (
    demographic_df
    .groupby("Characteristic")[
        "EHHWB n"
    ]
    .sum()
)

assert table_1_section_totals.nunique() == 1, (
    "Table 1 EHHWB counts do not sum to the same "
    "cohort size for every demographic section."
)

n_eligible_ehhwb_table_1 = int(
    table_1_section_totals.iloc[0]
)


# --------------------------------------------------
# Independently derive the same eligible EHHWB
# cohort from Supplementary Table S3
# --------------------------------------------------

s3_section_totals = (
    completion_by_demographic_df
    .groupby("Characteristic")[
        "n"
    ]
    .sum()
)

assert s3_section_totals.nunique() == 1, (
    "Supplementary Table S3 counts do not sum to "
    "the same cohort size for every section."
)

n_eligible_ehhwb_s3 = int(
    s3_section_totals.iloc[0]
)

assert (
    n_eligible_ehhwb_table_1
    == n_eligible_ehhwb_s3
), (
    "Table 1 and Supplementary Table S3 do not use "
    "the same eligible EHHWB respondent cohort."
)

n_eligible_ehhwb = (
    n_eligible_ehhwb_table_1
)


# --------------------------------------------------
# Derive EHHWB respondents without prerequisites
# --------------------------------------------------

n_ehhwb_without_prerequisites = (
    n_all_ehhwb
    - n_eligible_ehhwb
)

assert n_ehhwb_without_prerequisites >= 0, (
    "Eligible EHHWB respondents exceed all EHHWB "
    "respondents."
)


# --------------------------------------------------
# Derive ACE response cohorts from S1
# --------------------------------------------------

valid_response_distribution_df[
    "number_of_valid_ace_responses"
] = pd.to_numeric(
    valid_response_distribution_df[
        "number_of_valid_ace_responses"
    ],
    errors="raise"
)

valid_response_distribution_df[
    "participants_n"
] = pd.to_numeric(
    valid_response_distribution_df[
        "participants_n"
    ],
    errors="raise"
)


# Confirm that S1 includes the full EHHWB cohort.
n_s1_total = int(
    valid_response_distribution_df[
        "participants_n"
    ].sum()
)

assert n_s1_total == n_all_ehhwb, (
    "Supplementary Table S1 does not sum to the "
    "full EHHWB respondent cohort."
)


n_any_valid_ace = int(
    valid_response_distribution_df.loc[
        valid_response_distribution_df[
            "number_of_valid_ace_responses"
        ].gt(0),
        "participants_n"
    ].sum()
)


complete_11_rows = (
    valid_response_distribution_df.loc[
        valid_response_distribution_df[
            "number_of_valid_ace_responses"
        ].eq(11)
    ]
)

assert len(complete_11_rows) == 1, (
    "Supplementary Table S1 must contain exactly "
    "one row for 11 valid ACE responses."
)

n_complete_11_items = int(
    complete_11_rows[
        "participants_n"
    ].iloc[0]
)


# --------------------------------------------------
# Derive eight-domain cohort from Table 2
# --------------------------------------------------

eight_domain_distribution_df["n"] = (
    pd.to_numeric(
        eight_domain_distribution_df["n"],
        errors="raise"
    )
)

n_classifiable_8_domains = int(
    eight_domain_distribution_df[
        "n"
    ].sum()
)


# Confirm against the cumulative count at score 0.
score_zero_rows = (
    eight_domain_distribution_df.loc[
        eight_domain_distribution_df[
            "ACE Burden Score"
        ].eq(0)
    ]
)

if len(score_zero_rows) == 1:

    cumulative_n_at_zero = int(
        score_zero_rows[
            "Cumulative n (≥ score)"
        ].iloc[0]
    )

    assert (
        cumulative_n_at_zero
        == n_classifiable_8_domains
    ), (
        "The summed eight-domain cohort does not "
        "match the cumulative count at score 0."
    )


# ==================================================
# CALCULATE PERCENTAGES
# ==================================================

pct_prerequisites_of_survey = (
    100
    * n_completed_prerequisites
    / n_survey_conduct
)

pct_all_ehhwb_of_survey = (
    100
    * n_all_ehhwb
    / n_survey_conduct
)

pct_eligible_ehhwb_of_eligible = (
    100
    * n_eligible_ehhwb
    / n_completed_prerequisites
)

pct_without_prerequisites_of_ehhwb = (
    100
    * n_ehhwb_without_prerequisites
    / n_all_ehhwb
)

pct_any_valid_of_ehhwb = (
    100
    * n_any_valid_ace
    / n_all_ehhwb
)

pct_complete_11_of_ehhwb = (
    100
    * n_complete_11_items
    / n_all_ehhwb
)

pct_classifiable_8_of_ehhwb = (
    100
    * n_classifiable_8_domains
    / n_all_ehhwb
)


# ==================================================
# PRINT COHORT CHECKS
# ==================================================

print(
    f"Unique IDs in survey_conduct: "
    f"{n_survey_conduct:,}"
)

print(
    f"Completed all 3 prerequisites: "
    f"{n_completed_prerequisites:,}"
)

print(
    f"All EHHWB respondents: "
    f"{n_all_ehhwb:,}"
)

print(
    f"Prerequisite-eligible EHHWB respondents: "
    f"{n_eligible_ehhwb:,}"
)

print(
    f"EHHWB respondents without all prerequisites: "
    f"{n_ehhwb_without_prerequisites:,}"
)

print(
    f"At least 1 valid ACE item: "
    f"{n_any_valid_ace:,}"
)

print(
    f"Complete 11-item ACE response: "
    f"{n_complete_11_items:,}"
)

print(
    f"All 8 ACE domains classifiable: "
    f"{n_classifiable_8_domains:,}"
)


# ==================================================
# PREPARE DEMOGRAPHIC PANEL
# ==================================================

demo_plot_df = demographic_df.copy()

demo_plot_df["Characteristic"] = (
    demo_plot_df["Characteristic"]
    .replace({
        "Age Group": "Age"
    })
)

demo_plot_df["label"] = (
    demo_plot_df["Characteristic"].astype(str)
    + ": "
    + demo_plot_df["Category"].astype(str)
)


# Recover the sign of the standardized difference
# using the percentage-point difference.
demo_plot_df["standardized_difference"] = (
    np.sign(
        demo_plot_df["Diff, %"]
    )
    * demo_plot_df[
        "Abs. Stand. Mean Diff."
    ]
)


# Keep only demographic groups with |SMD| >= 0.15.
demo_plot_df = (
    demo_plot_df.loc[
        demo_plot_df[
            "Abs. Stand. Mean Diff."
        ].ge(0.15)
    ]
    .copy()
)


demo_plot_df = (
    demo_plot_df
    .sort_values(
        "standardized_difference",
        ascending=True
    )
    .reset_index(drop=True)
)


assert not demo_plot_df.empty, (
    "No demographic groups have an absolute "
    "standardized mean difference of at least 0.15."
)


display(
    demo_plot_df[
        [
            "Characteristic",
            "Category",
            "Eligible for EHHWB, %",
            "EHHWB, %",
            "standardized_difference"
        ]
    ]
)


# ==================================================
# FIGURE 1A: COHORT FLOW
# ==================================================

fig_a, ax1 = plt.subplots(
    figsize=(6.5, 7)
)

ax1.axis("off")


# Main vertical flow
main_boxes = [
    (
        0.5,
        0.91,
        (
            "Unique IDs in survey_conduct\n"
            f"n = {n_survey_conduct:,}"
        )
    ),

    (
        0.5,
        0.70,
        (
            "Eligible for EHHWB\n"
            "Completed all 3 prerequisite surveys\n"
            f"n = {n_completed_prerequisites:,}\n"
            f"{pct_prerequisites_of_survey:.1f}% of survey participants"
        )
    ),

    (
        0.5,
        0.49,
        (
            "Completed EHHWB survey\n"
            f"n = {n_all_ehhwb:,}\n"
            f"{pct_all_ehhwb_of_survey:.1f}% of survey participants"
        )
    ),

    (
        0.5,
        0.28,
        (
            "≥1 valid ACE item answered\n"
            f"n = {n_any_valid_ace:,}\n"
            f"{pct_any_valid_of_ehhwb:.1f}% of EHHWB respondents"
        )
    )
]


for x, y, text in main_boxes:
    ax1.text(
        x,
        y,
        text,
        ha="center",
        va="center",
        fontsize=9.8,
        bbox={
            "boxstyle": "round,pad=0.42",
            "edgecolor": "black",
            "facecolor": "white"
        }
    )


ax1.text(
    0.25,
    0.055,
    (
        "Complete 11-item questionnaire\n"
        "11/11 ACE items answered\n"
        f"n = {n_complete_11_items:,}\n"
        f"{pct_complete_11_of_ehhwb:.1f}% of EHHWB respondents"
    ),
    ha="center",
    va="center",
    fontsize=9.2,
    bbox={
        "boxstyle": "round,pad=0.42",
        "edgecolor": "black",
        "facecolor": "white"
    }
)


ax1.text(
    0.75,
    0.055,
    (
        "Eight-domain ACE cohort\n"
        "All 8 domains classifiable\n"
        f"n = {n_classifiable_8_domains:,}\n"
        f"{pct_classifiable_8_of_ehhwb:.1f}% of EHHWB respondents"
    ),
    ha="center",
    va="center",
    fontsize=9.2,
    bbox={
        "boxstyle": "round,pad=0.42",
        "edgecolor": "black",
        "facecolor": "white"
    }
)


for y_start, y_end in [
    (0.845, 0.775),
    (0.635, 0.565),
    (0.425, 0.355)
]:
    ax1.annotate(
        "",
        xy=(0.5, y_end),
        xytext=(0.5, y_start),
        arrowprops={
            "arrowstyle": "->",
            "lw": 1.5
        }
    )


ax1.annotate(
    "",
    xy=(0.25, 0.145),
    xytext=(0.47, 0.205),
    arrowprops={
        "arrowstyle": "->",
        "lw": 1.5
    }
)

ax1.annotate(
    "",
    xy=(0.75, 0.145),
    xytext=(0.53, 0.205),
    arrowprops={
        "arrowstyle": "->",
        "lw": 1.5
    }
)


ax1.set_title(
    "a Cohort assembly",
    fontsize=13,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
# ==================================================
# FIGURE 1B: DEMOGRAPHIC REPRESENTATIVENESS
# Run this in a new cell after the setup/Panel A cell
# ==================================================

fig_b, ax2 = plt.subplots(
    figsize=(8, 6)
)

ax2.barh(
    demo_plot_df["label"],
    demo_plot_df["standardized_difference"]
)

ax2.axvline(
    0,
    linewidth=1
)


# Set axis limits with room for value labels
minimum_difference = min(
    demo_plot_df["standardized_difference"].min(),
    0
)

maximum_difference = max(
    demo_plot_df["standardized_difference"].max(),
    0
)

difference_range = (
    maximum_difference
    - minimum_difference
)

if difference_range == 0:
    difference_range = 1

left_limit = (
    minimum_difference
    - 0.18 * difference_range
)

right_limit = (
    maximum_difference
    + 0.18 * difference_range
)

ax2.set_xlim(
    left_limit,
    right_limit
)


# Add signed SMD labels beside each bar
x_left, x_right = ax2.get_xlim()

label_gap = (
    0.015
    * (x_right - x_left)
)

for i, (_, row) in enumerate(
    demo_plot_df.iterrows()
):
    value = row["standardized_difference"]

    if value >= 0:
        x_text = value + label_gap
        horizontal_alignment = "left"
    else:
        x_text = value - label_gap
        horizontal_alignment = "right"

    if x_text >= x_right:
        x_text = x_right - label_gap
        horizontal_alignment = "right"
    elif x_text <= x_left:
        x_text = x_left + label_gap
        horizontal_alignment = "left"

    ax2.text(
        x_text,
        i,
        f"{value:+.2f}",
        va="center",
        ha=horizontal_alignment,
        fontsize=8
    )


ax2.set_xlabel(
    "Standardized difference\n"
    "EHHWB respondents relative to eligible cohort"
)

ax2.set_title(
    "b Demographic differences with |SMD| ≥ 0.15",
    fontsize=13,
    fontweight="bold"
)

ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ==================================================
# SAVE BOTH FIGURES
# ==================================================

# Save Figure 1A
fig_a.savefig(
    "outputs/figures/main/figure_1a_cohort_flow.png",
    dpi=300,
    bbox_inches="tight"
)

fig_a.savefig(
    "outputs/figures/main/figure_1a_cohort_flow.pdf",
    bbox_inches="tight"
)


# Save Figure 1B
fig_b.savefig(
    "outputs/figures/main/figure_1b_demographic_representativeness.png",
    dpi=300,
    bbox_inches="tight"
)

fig_b.savefig(
    "outputs/figures/main/figure_1b_demographic_representativeness.pdf",
    bbox_inches="tight"
)


print("Saved: outputs/figures/main/figure_1a_cohort_flow.png")
print("Saved: outputs/figures/main/figure_1a_cohort_flow.pdf")
print("Saved: outputs/figures/main/figure_1b_demographic_representativeness.png")
print("Saved: outputs/figures/main/figure_1b_demographic_representativeness.pdf")

## Supplementary Figure S1

Birth-year distribution of EHHWB-eligible participants and eligible EHHWB completers.

**Required upstream section:** Main Table 1. This figure uses the in-memory `table_1_raw` cohort, so run that section first in the current kernel.


In [ ]:
# ==================================================
# ADD YEAR OF BIRTH FROM PERSON TABLE
# ==================================================

birth_year_lookup = client.query(
    f"""
    SELECT
        person_id,
        year_of_birth
    FROM `{cdr}.person`
    """
).to_dataframe()


# Validate lookup
assert birth_year_lookup["person_id"].is_unique, (
    "The person table contains duplicate person IDs."
)


# Join birth year onto the existing eligible cohort
table_1_raw = (
    table_1_raw
    .drop(
        columns=["year_of_birth"],
        errors="ignore"
    )
    .merge(
        birth_year_lookup,
        on="person_id",
        how="left",
        validate="one_to_one"
    )
)


print(
    "Participants in eligible cohort: "
    f"{table_1_raw['person_id'].nunique():,}"
)

print(
    "Participants with available birth year: "
    f"{table_1_raw['year_of_birth'].notna().sum():,}"
)

display(
    table_1_raw[
        [
            "person_id",
            "completed_ehhwb",
            "year_of_birth"
        ]
    ].head()
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ==================================================
# SUPPLEMENTARY FIGURE S1:
# BIRTH-YEAR DISTRIBUTION OF EHHWB-ELIGIBLE
# PARTICIPANTS AND ELIGIBLE EHHWB COMPLETERS
# ==================================================

required_columns = [
    "person_id",
    "year_of_birth",
    "completed_ehhwb"
]

missing_columns = [
    column
    for column in required_columns
    if column not in table_1_raw.columns
]

assert not missing_columns, (
    "Missing required columns: "
    + ", ".join(missing_columns)
)


# --------------------------------------------------
# Prepare analytic dataset
# --------------------------------------------------

birth_year_df = (
    table_1_raw[required_columns]
    .copy()
)

birth_year_df["year_of_birth"] = pd.to_numeric(
    birth_year_df["year_of_birth"],
    errors="coerce"
)

assert birth_year_df["person_id"].is_unique, (
    "Eligible cohort contains duplicate person IDs."
)

assert birth_year_df["completed_ehhwb"].isin([0, 1]).all(), (
    "completed_ehhwb contains values other than 0 and 1."
)


# --------------------------------------------------
# Identify available and plausible birth years
# --------------------------------------------------

current_year = pd.Timestamp.today().year

birth_year_df["valid_birth_year"] = (
    birth_year_df["year_of_birth"].notna()
    & birth_year_df["year_of_birth"].between(
        1900,
        current_year
    )
)

n_invalid_birth_year = int(
    (~birth_year_df["valid_birth_year"]).sum()
)

print(
    "Participants with missing or implausible birth year: "
    f"{n_invalid_birth_year:,}"
)


birth_year_valid = (
    birth_year_df.loc[
        birth_year_df["valid_birth_year"]
    ]
    .copy()
)

birth_year_valid["year_of_birth"] = (
    birth_year_valid["year_of_birth"]
    .astype(int)
)


# ==================================================
# BIRTH-YEAR SUMMARY STATISTICS
# ==================================================

def summarize_birth_year(
    df,
    cohort_name
):
    values = (
        df["year_of_birth"]
        .dropna()
        .astype(int)
    )

    q1 = float(
        np.percentile(
            values,
            25,
            method="inverted_cdf"
        )
    )

    median = float(
        np.percentile(
            values,
            50,
            method="inverted_cdf"
        )
    )

    q3 = float(
        np.percentile(
            values,
            75,
            method="inverted_cdf"
        )
    )

    summary = {
        "Cohort": cohort_name,
        "N with birth year": int(values.size),
        "Minimum birth year": int(values.min()),
        "Q1": int(q1),
        "Median": int(median),
        "Q3": int(q3),
        "Maximum birth year": int(values.max()),
        "IQR": f"{int(q1)}–{int(q3)}",
        "Range": (
            f"{int(values.min())}–"
            f"{int(values.max())}"
        )
    }

    return summary


eligible_summary = summarize_birth_year(
    birth_year_valid,
    "EHHWB-eligible participants"
)

completer_summary = summarize_birth_year(
    birth_year_valid.loc[
        birth_year_valid["completed_ehhwb"].eq(1)
    ],
    "Eligible EHHWB completers"
)

birth_year_summary = pd.DataFrame(
    [
        eligible_summary,
        completer_summary
    ]
)

print("\nBirth-year summary statistics")
display(birth_year_summary)

for _, row in birth_year_summary.iterrows():
    print(
        f"{row['Cohort']}: "
        f"median = {row['Median']}; "
        f"IQR = {row['IQR']}; "
        f"range = {row['Range']}; "
        f"N = {row['N with birth year']:,}"
    )


# ==================================================
# CREATE BIRTH-DECADE CATEGORIES
# ==================================================

birth_year_bins = [
    -np.inf,
    1939,
    1949,
    1959,
    1969,
    1979,
    1989,
    1999,
    np.inf
]

birth_year_labels = [
    "<1940",
    "1940–1949",
    "1950–1959",
    "1960–1969",
    "1970–1979",
    "1980–1989",
    "1990–1999",
    "2000+"
]

birth_year_valid["birth_decade"] = pd.cut(
    birth_year_valid["year_of_birth"],
    bins=birth_year_bins,
    labels=birth_year_labels,
    include_lowest=True,
    right=True,
    ordered=True
)


assert birth_year_valid["birth_decade"].notna().all(), (
    "One or more valid birth years could not be categorized."
)


# ==================================================
# CALCULATE COUNTS AND WITHIN-COHORT PERCENTAGES
# ==================================================

n_eligible_birth_year = int(
    birth_year_valid["person_id"].nunique()
)

n_completer_birth_year = int(
    birth_year_valid.loc[
        birth_year_valid["completed_ehhwb"].eq(1),
        "person_id"
    ].nunique()
)


eligible_counts = (
    birth_year_valid
    .groupby(
        "birth_decade",
        observed=False
    )["person_id"]
    .nunique()
    .reindex(
        birth_year_labels,
        fill_value=0
    )
)

completer_counts = (
    birth_year_valid.loc[
        birth_year_valid["completed_ehhwb"].eq(1)
    ]
    .groupby(
        "birth_decade",
        observed=False
    )["person_id"]
    .nunique()
    .reindex(
        birth_year_labels,
        fill_value=0
    )
)


birth_decade_distribution = pd.DataFrame({
    "Birth decade": birth_year_labels,
    "Eligible n": eligible_counts.to_numpy(dtype=int),
    "Eligible completer n": completer_counts.to_numpy(dtype=int)
})

birth_decade_distribution["Eligible %"] = (
    100
    * birth_decade_distribution["Eligible n"]
    / n_eligible_birth_year
)

birth_decade_distribution["Eligible completer %"] = (
    100
    * birth_decade_distribution["Eligible completer n"]
    / n_completer_birth_year
)


# --------------------------------------------------
# All of Us privacy-policy check
# Do not display exact nonzero counts of 20 or fewer
# --------------------------------------------------

displayed_counts = np.concatenate([
    birth_decade_distribution["Eligible n"].to_numpy(),
    birth_decade_distribution[
        "Eligible completer n"
    ].to_numpy()
])

small_nonzero_counts = displayed_counts[
    (displayed_counts > 0)
    & (displayed_counts <= 20)
]

assert len(small_nonzero_counts) == 0, (
    "At least one plotted category has a count from 1 to 20. "
    "Collapse categories or suppress the exact count before "
    "exporting the figure under All of Us privacy policy."
)


print(
    "\nEligible participants with valid birth year: "
    f"{n_eligible_birth_year:,}"
)

print(
    "Eligible EHHWB completers with valid birth year: "
    f"{n_completer_birth_year:,}"
)

display(
    birth_decade_distribution.round({
        "Eligible %": 2,
        "Eligible completer %": 2
    })
)


# ==================================================
# SET FONTS
# ==================================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [
        "Helvetica",
        "Arial",
        "Liberation Sans",
        "DejaVu Sans"
    ],
    "mathtext.fontset": "dejavusans",
    "axes.unicode_minus": False
})


# ==================================================
# CREATE SUPPLEMENTARY FIGURE S1
# ==================================================

fig_s1, ax_s1 = plt.subplots(
    figsize=(11, 6)
)

x = np.arange(
    len(birth_year_labels)
)

bar_width = 0.38

eligible_bars = ax_s1.bar(
    x - bar_width / 2,
    birth_decade_distribution["Eligible %"],
    width=bar_width,
    label="EHHWB-eligible participants"
)

completer_bars = ax_s1.bar(
    x + bar_width / 2,
    birth_decade_distribution[
        "Eligible completer %"
    ],
    width=bar_width,
    label="Eligible EHHWB completers"
)


# --------------------------------------------------
# Add exact n above each bar
# --------------------------------------------------

def add_count_labels(
    ax,
    bars,
    counts
):
    for bar, count in zip(
        bars,
        counts
    ):
        height = bar.get_height()

        ax.annotate(
            f"n={int(count):,}",
            xy=(
                bar.get_x() + bar.get_width() / 2,
                height
            ),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            rotation=90,
            fontsize=8
        )


add_count_labels(
    ax_s1,
    eligible_bars,
    birth_decade_distribution["Eligible n"]
)

add_count_labels(
    ax_s1,
    completer_bars,
    birth_decade_distribution[
        "Eligible completer n"
    ]
)


# --------------------------------------------------
# Format axes and title
# --------------------------------------------------

maximum_percentage = float(
    birth_decade_distribution[
        [
            "Eligible %",
            "Eligible completer %"
        ]
    ]
    .to_numpy()
    .max()
)

ax_s1.set_xticks(
    x
)

ax_s1.set_xticklabels(
    birth_year_labels,
    rotation=35,
    ha="right"
)

ax_s1.set_ylim(
    0,
    maximum_percentage * 1.28
)

ax_s1.set_xlabel(
    "Birth year"
)

ax_s1.set_ylabel(
    "Percentage of participants"
)

ax_s1.set_title(
    "Birth-year distribution of "
    "EHHWB-eligible participants and eligible EHHWB completers",
    fontsize=13,
    fontweight="bold"
)

ax_s1.legend(
    frameon=False
)

ax_s1.spines["top"].set_visible(False)
ax_s1.spines["right"].set_visible(False)

fig_s1.tight_layout()

plt.show()


# ==================================================
# OPTIONAL: SAVE FIGURE AND SOURCE DATA
# ==================================================

fig_s1.savefig(
    "outputs/figures/supplementary/supplementary_figure_s1_birth_year_distribution.png",
    dpi=600,
    bbox_inches="tight"
)

fig_s1.savefig(
    "outputs/figures/supplementary/supplementary_figure_s1_birth_year_distribution.pdf",
    bbox_inches="tight"
)

birth_decade_distribution.to_csv(
    "outputs/figure_data/supplementary/supplementary_figure_s1_birth_year_distribution.csv",
    index=False
)

## Main Figure 2

Distribution of eight-domain ACE burden scores.

**Required input:** Main Table 2. Run that section first if its exported CSV file is not present.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ==================================================
# FIGURE 2:
# DISTRIBUTION OF 8-DOMAIN ACE BURDEN SCORES
# ==================================================

ace_distribution_file = (
    "outputs/tables/main/table_2_distribution_of_8_domain_ace_burden_scores.csv"
)

ace_dist = pd.read_csv(
    ace_distribution_file
)


# --------------------------------------------------
# Validate and standardize columns
# --------------------------------------------------

required_columns = [
    "ACE Burden Score",
    "n",
    "%",
    "Cumulative n (≥ score)",
    "Cumulative % (≥ score)"
]

missing_columns = [
    column
    for column in required_columns
    if column not in ace_dist.columns
]

assert not missing_columns, (
    "Missing required columns: "
    + ", ".join(missing_columns)
)


ace_dist = (
    ace_dist[required_columns]
    .copy()
    .sort_values("ACE Burden Score")
    .reset_index(drop=True)
)

ace_dist["ACE Burden Score"] = (
    ace_dist["ACE Burden Score"]
    .astype(int)
)

ace_dist["n"] = (
    ace_dist["n"]
    .astype(int)
)

assert ace_dist[
    "ACE Burden Score"
].tolist() == list(range(0, 9)), (
    "ACE burden scores are not exactly 0 through 8."
)


# ==================================================
# CALCULATE EMPIRICAL CUMULATIVE DISTRIBUTION
# ==================================================

n_total = int(
    ace_dist["n"].sum()
)

ace_dist["Cumulative n (≤ score)"] = (
    ace_dist["n"]
    .cumsum()
)

ace_dist["Cumulative % (≤ score)"] = (
    100
    * ace_dist["Cumulative n (≤ score)"]
    / n_total
)


# ==================================================
# SUMMARY STATISTICS
# ==================================================

expanded_scores = np.repeat(
    ace_dist["ACE Burden Score"].to_numpy(),
    ace_dist["n"].to_numpy()
)

mean_ace = float(
    np.mean(expanded_scores)
)

median_ace = float(
    np.median(expanded_scores)
)

sd_ace = float(
    np.std(
        expanded_scores,
        ddof=1
    )
)

percentile_values = {
    "25th": float(
        np.percentile(
            expanded_scores,
            25,
            method="inverted_cdf"
        )
    ),
    "50th": float(
        np.percentile(
            expanded_scores,
            50,
            method="inverted_cdf"
        )
    ),
    "75th": float(
        np.percentile(
            expanded_scores,
            75,
            method="inverted_cdf"
        )
    ),
    "90th": float(
        np.percentile(
            expanded_scores,
            90,
            method="inverted_cdf"
        )
    ),
    "95th": float(
        np.percentile(
            expanded_scores,
            95,
            method="inverted_cdf"
        )
    )
}


print(f"N = {n_total:,}")
print(f"Mean ACE burden score = {mean_ace:.2f}")
print(f"Median ACE burden score = {median_ace:.0f}")
print(f"SD = {sd_ace:.2f}")
print(percentile_values)

display(ace_dist)


# ==================================================
# SET FONTS
# ==================================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [
        "Helvetica",
        "Arial",
        "Liberation Sans",
        "DejaVu Sans"
    ],
    "mathtext.fontset": "dejavusans",
    "axes.unicode_minus": False
})


# ==================================================
# PANEL A:
# DISTRIBUTION OF ACE BURDEN SCORES
# ==================================================

fig_a, ax_a = plt.subplots(
    figsize=(8, 5)
)

ax_a.bar(
    ace_dist["ACE Burden Score"],
    ace_dist["n"]
)

ax_a.set_xticks(
    range(0, 9)
)

ax_a.set_xlabel(
    "8-domain ACE burden score"
)

ax_a.set_ylabel(
    "Number of participants"
)

ax_a.set_title(
    "a Distribution of 8-domain ACE burden scores\n"
    f"N = {n_total:,}",
    fontsize=13,
    fontweight="bold"
)

ax_a.text(
    0.98,
    0.95,
    (
        f"Mean = {mean_ace:.2f}\n"
        f"Median = {median_ace:.0f}\n"
        f"SD = {sd_ace:.2f}"
    ),
    transform=ax_a.transAxes,
    ha="right",
    va="top",
    fontsize=10
)

ax_a.spines["top"].set_visible(False)
ax_a.spines["right"].set_visible(False)

fig_a.tight_layout()

plt.show()


# ==================================================
# PANEL B:
# EMPIRICAL CUMULATIVE DISTRIBUTION
# ==================================================

fig_b, ax_b = plt.subplots(
    figsize=(8, 5)
)

ax_b.plot(
    ace_dist["ACE Burden Score"],
    ace_dist["Cumulative % (≤ score)"],
    marker="o",
    linewidth=1.8
)


# --------------------------------------------------
# Add percentile reference lines only
# --------------------------------------------------

for label, score in percentile_values.items():

    ax_b.axvline(
        score,
        linestyle="--",
        linewidth=1
    )

    ax_b.text(
        score + 0.05,
        4,
        label,
        rotation=90,
        va="bottom",
        ha="left",
        fontsize=9
    )


ax_b.set_xticks(
    range(0, 9)
)

ax_b.set_yticks(
    range(0, 101, 10)
)

ax_b.set_xlim(
    -0.4,
    8.4
)

ax_b.set_ylim(
    0,
    103
)

ax_b.set_xlabel(
    "8-domain ACE burden score"
)

ax_b.set_ylabel(
    "Cumulative percentage"
)

ax_b.set_title(
    "b Empirical cumulative distribution of "
    "8-domain ACE burden scores",
    fontsize=13,
    fontweight="bold"
)

ax_b.spines["top"].set_visible(False)
ax_b.spines["right"].set_visible(False)

fig_b.tight_layout()

plt.show()

In [ ]:
# ==================================================
# SAVE FIGURES
# ==================================================

fig_a.savefig(
    "outputs/figures/main/figure_2a_distribution_of_8_domain_ace_burden_scores.png",
    dpi=300,
    bbox_inches="tight"
)

fig_a.savefig(
    "outputs/figures/main/figure_2a_distribution_of_8_domain_ace_burden_scores.pdf",
    bbox_inches="tight"
)

fig_b.savefig(
    "outputs/figures/main/figure_2b_empirical_cumulative_distribution_of_8_domain_ace_burden_scores.png",
    dpi=300,
    bbox_inches="tight"
)

fig_b.savefig(
    "outputs/figures/main/figure_2b_empirical_cumulative_distribution_of_8_domain_ace_burden_scores.pdf",
    bbox_inches="tight"
)

print(
    "Saved: "
    "outputs/figures/main/figure_2a_distribution_of_8_domain_ace_burden_scores.png"
)

print(
    "Saved: "
    "outputs/figures/main/figure_2a_distribution_of_8_domain_ace_burden_scores.pdf"
)

print(
    "Saved: "
    "outputs/figures/main/figure_2b_empirical_cumulative_distribution_of_8_domain_ace_burden_scores.png"
)

print(
    "Saved: "
    "outputs/figures/main/figure_2b_empirical_cumulative_distribution_of_8_domain_ace_burden_scores.pdf"
)

## Demographic differences in ACE burden


In [ ]:
from google.cloud import bigquery
import pandas as pd


# ==================================================
# ADD DEMOGRAPHICS TO PERSON-LEVEL ACE SCORES
# ==================================================

ehhwb_survey_concept_id = 1703970

ace_demographics = client.query(
    f"""
    WITH earliest_ehhwb AS (
        SELECT
            person_id,
            survey_end_date
        FROM `{cdr}.survey_conduct`
        WHERE survey_source_concept_id = @ehhwb_survey_concept_id
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY person_id
            ORDER BY
                survey_end_date ASC,
                survey_conduct_id ASC
        ) = 1
    )

    SELECT
        e.person_id,

        race.concept_name AS race,
        ethnicity.concept_name AS ethnicity,
        gender.concept_name AS gender,

        EXTRACT(YEAR FROM e.survey_end_date)
            - p.year_of_birth AS age_at_ehhwb,

        CASE
            WHEN p.year_of_birth IS NULL
                OR e.survey_end_date IS NULL
                THEN 'Unknown'

            WHEN EXTRACT(YEAR FROM e.survey_end_date)
                - p.year_of_birth BETWEEN 18 AND 30
                THEN '18-30'

            WHEN EXTRACT(YEAR FROM e.survey_end_date)
                - p.year_of_birth BETWEEN 31 AND 45
                THEN '31-45'

            WHEN EXTRACT(YEAR FROM e.survey_end_date)
                - p.year_of_birth BETWEEN 46 AND 60
                THEN '46-60'

            WHEN EXTRACT(YEAR FROM e.survey_end_date)
                - p.year_of_birth BETWEEN 61 AND 75
                THEN '61-75'

            WHEN EXTRACT(YEAR FROM e.survey_end_date)
                - p.year_of_birth >= 76
                THEN '76+'

            ELSE 'Unknown'
        END AS age_group

    FROM earliest_ehhwb e

    INNER JOIN `{cdr}.person` p
        ON e.person_id = p.person_id

    LEFT JOIN `{cdr}.concept` race
        ON p.race_concept_id = race.concept_id

    LEFT JOIN `{cdr}.concept` ethnicity
        ON p.ethnicity_concept_id = ethnicity.concept_id

    LEFT JOIN `{cdr}.concept` gender
        ON p.gender_concept_id = gender.concept_id
    """,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter(
                "ehhwb_survey_concept_id",
                "INT64",
                ehhwb_survey_concept_id
            )
        ]
    ),
).to_dataframe()


# ==================================================
# MERGE ACE SCORES WITH DEMOGRAPHICS
# ==================================================

ace_with_demographics = (
    ace_8_domain_person_level
    .merge(
        ace_demographics,
        on="person_id",
        how="left",
        validate="one_to_one"
    )
)


# Restrict to participants with a complete 8-domain score
complete_ace_with_demographics = (
    ace_with_demographics
    .loc[
        ace_with_demographics["ace_burden_score"].notna()
    ]
    .copy()
)

complete_ace_with_demographics["ace_burden_score"] = (
    complete_ace_with_demographics[
        "ace_burden_score"
    ].astype(float)
)

n_complete = (
    complete_ace_with_demographics["person_id"].nunique()
)

print(
    f"Participants with complete 8-domain ACE scores: "
    f"{n_complete:,}"
)

## Supplementary Table S5

ACE burden scores by demographic characteristics.


In [ ]:
# ==================================================
# SUPPLEMENTARY TABLE S5:
# ACE BURDEN BY DEMOGRAPHIC CHARACTERISTICS
# ==================================================

def build_ace_demographic_section(
    df,
    group_col,
    characteristic_label,
    category_order=None
):
    temp = df.copy()

    temp[group_col] = (
        temp[group_col]
        .astype("object")
        .where(
            temp[group_col].notna(),
            "Missing"
        )
    )

    section = (
        temp
        .groupby(
            group_col,
            dropna=False
        )
        .agg(
            participants_n=(
                "person_id",
                "nunique"
            ),
            mean_ace_burden_score=(
                "ace_burden_score",
                "mean"
            ),
            median_ace_burden_score=(
                "ace_burden_score",
                "median"
            ),
            std_ace_burden_score=(
                "ace_burden_score",
                "std"
            )
        )
        .reset_index()
        .rename(
            columns={
                group_col: "Name"
            }
        )
    )

    section.insert(
        0,
        "Cohort",
        "Complete"
    )

    section.insert(
        0,
        "Characteristic",
        characteristic_label
    )

    if category_order is not None:
        observed_categories = set(
            section["Name"].astype(str)
        )

        ordered_categories = [
            category
            for category in category_order
            if category in observed_categories
        ]

        additional_categories = [
            category
            for category in section["Name"].astype(str)
            if category not in ordered_categories
        ]

        final_order = (
            ordered_categories
            + additional_categories
        )

        section["Name"] = pd.Categorical(
            section["Name"].astype(str),
            categories=final_order,
            ordered=True
        )

        section = (
            section
            .sort_values("Name")
            .reset_index(drop=True)
        )

        section["Name"] = (
            section["Name"].astype(str)
        )

    else:
        section = (
            section
            .sort_values(
                "participants_n",
                ascending=False
            )
            .reset_index(drop=True)
        )

    return section


# --------------------------------------------------
# Build demographic sections
# --------------------------------------------------

race_section = build_ace_demographic_section(
    complete_ace_with_demographics,
    group_col="race",
    characteristic_label="Race"
)

ethnicity_section = build_ace_demographic_section(
    complete_ace_with_demographics,
    group_col="ethnicity",
    characteristic_label="Ethnicity"
)

gender_section = build_ace_demographic_section(
    complete_ace_with_demographics,
    group_col="gender",
    characteristic_label="Gender"
)

age_section = build_ace_demographic_section(
    complete_ace_with_demographics,
    group_col="age_group",
    characteristic_label="Age Group",
    category_order=[
        "18-30",
        "31-45",
        "46-60",
        "61-75",
        "76+",
        "Unknown",
        "Missing"
    ]
)


# --------------------------------------------------
# Combine sections
# --------------------------------------------------

supplementary_table_s5 = pd.concat(
    [
        race_section,
        ethnicity_section,
        gender_section,
        age_section
    ],
    ignore_index=True
)


# --------------------------------------------------
# Format columns
# --------------------------------------------------

supplementary_table_s5 = (
    supplementary_table_s5
    .rename(
        columns={
            "participants_n": "Participants (n)",
            "mean_ace_burden_score": (
                "Mean ACE Burden Score"
            ),
            "median_ace_burden_score": (
                "Median ACE Burden Score"
            ),
            "std_ace_burden_score": (
                "Standard Deviation"
            )
        }
    )
)

supplementary_table_s5[
    [
        "Mean ACE Burden Score",
        "Median ACE Burden Score",
        "Standard Deviation"
    ]
] = supplementary_table_s5[
    [
        "Mean ACE Burden Score",
        "Median ACE Burden Score",
        "Standard Deviation"
    ]
].round(2)

supplementary_table_s5 = supplementary_table_s5[
    [
        "Characteristic",
        "Cohort",
        "Name",
        "Participants (n)",
        "Mean ACE Burden Score",
        "Median ACE Burden Score",
        "Standard Deviation"
    ]
]


# ==================================================
# VALIDATION
# ==================================================

for characteristic in [
    "Race",
    "Ethnicity",
    "Gender",
    "Age Group"
]:
    section_total = supplementary_table_s5.loc[
        supplementary_table_s5[
            "Characteristic"
        ].eq(characteristic),
        "Participants (n)"
    ].sum()

    assert section_total == n_complete, (
        f"{characteristic} counts sum to "
        f"{section_total:,}, not {n_complete:,}."
    )

    print(
        f"{characteristic} section total: "
        f"{int(section_total):,}"
    )


display(supplementary_table_s5)

In [ ]:
# ==================================================
# SAVE OUTPUTS
# ==================================================

supplementary_table_s5.to_csv(
    "outputs/tables/supplementary/supplementary_table_s5_ace_burden_scores_by_demographic_characteristics.csv",
    index=False
)

print(
    "Saved: "
    "outputs/tables/supplementary/supplementary_table_s5_ace_burden_scores_by_demographic_characteristics.csv"
)

## Main Figure 3

Demographic differences in eight-domain ACE burden scores.

**Required input:** Supplementary Table S5. Run that section first if its exported CSV file is not present.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Figure 3: Demographic differences in eight-domain ACE burden scores
# -----------------------------

# -----------------------------
# Load demographic ACE summary
# -----------------------------

demo_plot = pd.read_csv(
    "outputs/tables/supplementary/supplementary_table_s5_ace_burden_scores_by_demographic_characteristics.csv"
)

display(demo_plot)

contains_prefer_not = demo_plot["Name"].str.contains(
    "prefer not",
    case=False,
    na=False
)


# -----------------------------
# Clean demographic labels
# -----------------------------

label_replacements = {
    # Race
    "None of these": "None of these races",

    # Ethnicity
    "What Race Ethnicity: Race Ethnicity None Of These":
        "None of these ethnicities",
    "PMI: Prefer Not To Answer":
        "I prefer not to answer",

    # Gender
    "Not man only, not woman only, prefer not to answer, or skipped":
        "Not man only, not woman only",
    "Gender Identity: Non Binary":
        "Non-binary",
    "Gender Identity: Transgender":
        "Transgender",
    "Gender Identity: Additional Options":
        "Additional Options"
}

demo_plot["Name"] = demo_plot["Name"].replace(label_replacements)


# -----------------------------
# Optional exclusions
# -----------------------------
# Exclude administrative or very small/undefined categories from the figure.
# They remain available in the supplementary table.

excluded_groups = [
    "PMI: Skip",
    "No matching concept"
]

demo_plot = demo_plot[
    ~demo_plot["Name"].isin(excluded_groups)
    & ~contains_prefer_not
].copy()


# -----------------------------
# Helpful ordering
# -----------------------------

variable_order = [
    "Race",
    "Ethnicity",
    "Gender",
    "Age Group"
]

variable_order_map = {
    variable: order
    for order, variable in enumerate(variable_order)
}

demo_plot["variable_order"] = (
    demo_plot["Characteristic"]
    .map(variable_order_map)
)

demo_plot = demo_plot.sort_values(
    ["variable_order", "Mean ACE Burden Score"],
    ascending=[True, False]
)


# -----------------------------
# Mean ACE burden by demographic group
# -----------------------------

fig, ax = plt.subplots(figsize=(9, 8))

ax.barh(
    demo_plot["Name"],
    demo_plot["Mean ACE Burden Score"]
)

ax.set_xlabel("Mean ACE Burden")
ax.set_ylabel("")
ax.set_title("Mean ACE Burden by Demographic Group")

ax.invert_yaxis()

fig.tight_layout()
plt.show()

In [ ]:
# ==================================================
# SAVE FIGURE
# ==================================================

fig.savefig(
    "outputs/figures/main/figure_3_demographic_differences_in_8_domain_ace_burden_scores.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    "outputs/figures/main/figure_3_demographic_differences_in_8_domain_ace_burden_scores.pdf",
    bbox_inches="tight"
)

## Supplementary Table S6

Demographic predictors of elevated eight-domain ACE burden.


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
import re


# ==================================================
# SETTINGS
# ==================================================

thresholds = {
    2: "ACE ≥ 2",
    4: "ACE ≥ 4",
    7: "ACE ≥ 7"
}

reference_categories = {
    "age_group": "61-75",
    "gender": "Female",
    "race": "White",
    "ethnicity": "Not Hispanic or Latino"
}

characteristic_labels = {
    "age_group": "Age Group",
    "gender": "Gender",
    "race": "Race",
    "ethnicity": "Ethnicity"
}


# ==================================================
# PREPARE MODEL DATA
# ==================================================

s6_model_data = complete_ace_with_demographics[
    [
        "person_id",
        "ace_burden_score",
        "age_group",
        "gender",
        "race",
        "ethnicity"
    ]
].copy()

for column in [
    "age_group",
    "gender",
    "race",
    "ethnicity"
]:
    s6_model_data[column] = (
        s6_model_data[column]
        .astype("object")
        .where(
            s6_model_data[column].notna(),
            "Missing"
        )
        .astype(str)
    )

s6_model_data["age_group"] = (
    s6_model_data["age_group"]
    .replace({
        "18-30": "<=30",
        "≤30": "<=30"
    })
)


# ==================================================
# FORMATTERS
# ==================================================

def format_p_value(p_value):

    if pd.isna(p_value) or not np.isfinite(p_value):
        return "NE"

    if p_value < 0.001:
        return "<0.001"

    return f"{p_value:.4f}".rstrip("0").rstrip(".")


def format_or_ci(
    odds_ratio,
    ci_lower,
    ci_upper
):

    if not all(
        np.isfinite(value)
        for value in [
            odds_ratio,
            ci_lower,
            ci_upper
        ]
    ):
        return "NE"

    return (
        f"{odds_ratio:.3f} "
        f"({ci_lower:.3f}–{ci_upper:.3f})"
    )


def parse_parameter(parameter_name):
    """
    Parses statsmodels coefficient names such as:

    C(age_group, Treatment(reference='61-75'))[T.31-45]

    or

    C(age_group, Treatment(reference="61-75"))[T.31-45]
    """

    if parameter_name == "Intercept":
        return None, None

    match = re.match(
        r"C\(([^,]+),\s*Treatment\(reference=['\"].*?['\"]\)\)"
        r"\[T\.(.*)\]$",
        parameter_name
    )

    if match is None:
        return None, None

    variable = match.group(1).strip()
    category = match.group(2).strip()

    return variable, category

In [ ]:
# ==================================================
# FIT BINOMIAL GLM MODELS
# ==================================================

formula = """
    elevated_ace
    ~ C(age_group, Treatment(reference="61-75"))
    + C(gender, Treatment(reference="Female"))
    + C(race, Treatment(reference="White"))
    + C(
        ethnicity,
        Treatment(reference="Not Hispanic or Latino")
      )
"""

all_estimates = []
s6_rows = []

for threshold_value, threshold_label in thresholds.items():

    model_df = s6_model_data.copy()

    model_df["elevated_ace"] = (
        model_df["ace_burden_score"]
        .ge(threshold_value)
        .astype(int)
    )

    model_n = len(model_df)
    cases_n = int(model_df["elevated_ace"].sum())
    controls_n = int(model_n - cases_n)
    cases_pct = 100 * cases_n / model_n

    # --------------------------------------------------
    # Check reference categories
    # --------------------------------------------------

    for variable, reference in reference_categories.items():

        assert reference in model_df[variable].unique(), (
            f'Reference "{reference}" is missing from '
            f'"{variable}".'
        )

    # --------------------------------------------------
    # Fit adjusted logistic regression using GLM
    # --------------------------------------------------

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        model_result = smf.glm(
            formula=formula,
            data=model_df,
            family=sm.families.Binomial()
        ).fit(
            maxiter=1000
        )

    print(
        f"{threshold_label}: "
        f"converged = {model_result.converged}; "
        f"N = {int(model_result.nobs):,}"
    )

    # --------------------------------------------------
    # Extract coefficients
    # --------------------------------------------------

    confidence_intervals = model_result.conf_int()

    estimates = pd.DataFrame({
        "parameter": model_result.params.index,
        "coefficient": model_result.params.values,
        "standard_error": model_result.bse.values,
        "p_value": model_result.pvalues.values,
        "ci_lower_log": confidence_intervals.iloc[:, 0].values,
        "ci_upper_log": confidence_intervals.iloc[:, 1].values
    })

    parsed = estimates["parameter"].apply(
        parse_parameter
    )

    estimates["variable"] = [
        result[0] for result in parsed
    ]

    estimates["name"] = [
        result[1] for result in parsed
    ]

    estimates = estimates.loc[
        estimates["variable"].notna()
    ].copy()

    estimates["odds_ratio"] = np.exp(
        estimates["coefficient"]
    )

    estimates["ci_lower"] = np.exp(
        estimates["ci_lower_log"]
    )

    estimates["ci_upper"] = np.exp(
        estimates["ci_upper_log"]
    )

    estimates["Threshold"] = threshold_label

    all_estimates.append(estimates.copy())

    # --------------------------------------------------
    # Build output rows
    # --------------------------------------------------

    for _, estimate in estimates.iterrows():

        finite_estimate = all(
            np.isfinite(value)
            for value in [
                estimate["coefficient"],
                estimate["standard_error"],
                estimate["odds_ratio"],
                estimate["ci_lower"],
                estimate["ci_upper"],
                estimate["p_value"]
            ]
        )

        if finite_estimate:

            or_ci = format_or_ci(
                estimate["odds_ratio"],
                estimate["ci_lower"],
                estimate["ci_upper"]
            )

            p_value = format_p_value(
                estimate["p_value"]
            )

        else:

            or_ci = "NE"
            p_value = "NE"

        variable = estimate["variable"]

        s6_rows.append({
            "Threshold": threshold_label,
            "Category": characteristic_labels[variable],
            "Name": estimate["name"],
            "Reference": reference_categories[variable],
            "Model, N": model_n,
            "Cases, N": cases_n,
            "Controls, N": controls_n,
            "Cases, %": cases_pct,
            "OR (95% CI)": or_ci,
            "p-value": p_value
        })


# ==================================================
# CREATE FINAL TABLE
# ==================================================

supplementary_table_s6 = pd.DataFrame(s6_rows)

supplementary_table_s6["Cases, %"] = (
    supplementary_table_s6["Cases, %"]
    .round(2)
)

supplementary_table_s6 = supplementary_table_s6[
    [
        "Threshold",
        "Category",
        "Name",
        "Reference",
        "Model, N",
        "Cases, N",
        "Controls, N",
        "Cases, %",
        "OR (95% CI)",
        "p-value"
    ]
]

display(supplementary_table_s6)

In [ ]:
# ==================================================
# DIAGNOSTICS
# ==================================================

diagnostic_estimates = pd.concat(
    all_estimates,
    ignore_index=True
)

display(
    diagnostic_estimates[
        [
            "Threshold",
            "variable",
            "name",
            "coefficient",
            "standard_error",
            "odds_ratio",
            "ci_lower",
            "ci_upper",
            "p_value"
        ]
    ].head(30)
)

print("\nOR result counts:")

print(
    supplementary_table_s6[
        "OR (95% CI)"
    ].value_counts(dropna=False)
)

In [ ]:
# ==================================================
# SAVE OUTPUTS
# ==================================================

supplementary_table_s6.to_csv(
    "outputs/tables/supplementary/supplementary_table_s6_demographic_predictors_of_elevated_8_domain_ace_burden.csv",
    index=False
)

print(
    "Saved: "
    "outputs/tables/supplementary/supplementary_table_s6_demographic_predictors_of_elevated_8_domain_ace_burden.csv"
)

## Supplementary Table S8

Internal consistency of the eight-domain ACE score.


In [ ]:
import numpy as np
import pandas as pd


# ==================================================
# SUPPLEMENTARY TABLE S8:
# INTERNAL CONSISTENCY OF THE 8-DOMAIN ACE SCORE
# ==================================================

domain_labels = {
    "household_mental_illness": (
        "Household mental illness"
    ),
    "household_substance_use": (
        "Household substance use"
    ),
    "incarcerated_household_member": (
        "Incarcerated household member"
    ),
    "parental_separation_divorce": (
        "Parental separation or divorce"
    ),
    "witnessed_household_violence": (
        "Witnessed violence between adults in the home"
    ),
    "physical_abuse": (
        "Physical abuse"
    ),
    "emotional_abuse": (
        "Emotional abuse"
    ),
    "sexual_abuse": (
        "Sexual abuse"
    )
}

domain_columns = list(domain_labels.keys())


# --------------------------------------------------
# Restrict to participants with all 8 domains
# classifiable
# --------------------------------------------------

complete_domain_data = (
    ace_8_domain_person_level
    .loc[
        ace_8_domain_person_level[
            domain_columns
        ].notna().all(axis=1),
        ["person_id"] + domain_columns
    ]
    .copy()
)

complete_domain_data[domain_columns] = (
    complete_domain_data[domain_columns]
    .astype(int)
)

n_complete = len(complete_domain_data)

print(
    "Participants with all 8 domains classifiable: "
    f"{n_complete:,}"
)

In [ ]:
# ==================================================
# KR-20 FUNCTION
# ==================================================

def calculate_kr20(binary_df):
    """
    Calculate Kuder-Richardson Formula 20 for binary items.

    Parameters
    ----------
    binary_df : pandas.DataFrame
        DataFrame containing only binary 0/1 variables.

    Returns
    -------
    float
        KR-20 reliability coefficient.
    """

    k = binary_df.shape[1]

    if k < 2:
        return np.nan

    item_proportions = binary_df.mean(axis=0)

    item_variances = (
        item_proportions
        * (1 - item_proportions)
    )

    total_score = binary_df.sum(axis=1)

    total_score_variance = total_score.var(
        ddof=1
    )

    if total_score_variance == 0:
        return np.nan

    kr20 = (
        k / (k - 1)
        * (
            1
            - item_variances.sum()
            / total_score_variance
        )
    )

    return kr20


# ==================================================
# FULL 8-DOMAIN KR-20
# ==================================================

full_kr20 = calculate_kr20(
    complete_domain_data[domain_columns]
)

print(
    f"Full 8-domain KR-20: {full_kr20:.3f}"
)

In [ ]:
# ==================================================
# BUILD SUPPLEMENTARY TABLE S8
# ==================================================

s8_rows = []

for domain in domain_columns:

    domain_values = complete_domain_data[domain]

    positive_n = int(
        domain_values.eq(1).sum()
    )

    total_n = int(
        domain_values.notna().sum()
    )

    positive_pct = (
        100 * positive_n / total_n
        if total_n > 0
        else np.nan
    )

    # --------------------------------------------------
    # Corrected item-total correlation
    #
    # Correlate the domain with the sum of all OTHER
    # domains, excluding the domain itself.
    # --------------------------------------------------

    other_domains = [
        column
        for column in domain_columns
        if column != domain
    ]

    corrected_total_score = (
        complete_domain_data[
            other_domains
        ].sum(axis=1)
    )

    corrected_item_total_r = (
        domain_values.corr(
            corrected_total_score
        )
    )

    # --------------------------------------------------
    # KR-20 if this domain were deleted
    # --------------------------------------------------

    kr20_if_deleted = calculate_kr20(
        complete_domain_data[
            other_domains
        ]
    )

    # Negative values mean reliability decreases
    # after deletion.
    delta_kr20 = (
        kr20_if_deleted
        - full_kr20
    )

    s8_rows.append({
        "Domain": domain_labels[domain],
        "Positive Response, n": positive_n,
        "Total, n": total_n,
        "Positive Response, %": positive_pct,
        "Corrected item-total r": corrected_item_total_r,
        "KR-20 if deleted": kr20_if_deleted,
        "ΔKR-20": delta_kr20
    })


supplementary_table_s8 = pd.DataFrame(
    s8_rows
)


# --------------------------------------------------
# Round output
# --------------------------------------------------

supplementary_table_s8[
    "Positive Response, %"
] = (
    supplementary_table_s8[
        "Positive Response, %"
    ]
    .round(2)
)

supplementary_table_s8[
    [
        "Corrected item-total r",
        "KR-20 if deleted",
        "ΔKR-20"
    ]
] = supplementary_table_s8[
    [
        "Corrected item-total r",
        "KR-20 if deleted",
        "ΔKR-20"
    ]
].round(3)


display(supplementary_table_s8)

In [ ]:
# ==================================================
# VALIDATION
# ==================================================

assert len(supplementary_table_s8) == 8

assert (
    supplementary_table_s8["Total, n"]
    .eq(n_complete)
    .all()
), (
    "At least one domain has a denominator that does "
    "not equal the complete-case cohort."
)

assert complete_domain_data[
    domain_columns
].isin([0, 1]).all().all(), (
    "At least one domain contains values other than 0 or 1."
)

assert np.isclose(
    complete_domain_data[
        domain_columns
    ].sum(axis=1),
    ace_8_domain_person_level.loc[
        ace_8_domain_person_level[
            "ace_burden_score"
        ].notna(),
        "ace_burden_score"
    ].to_numpy()
).all(), (
    "The sum of the 8 domains does not match "
    "ace_burden_score."
)

print(
    f"Full 8-domain KR-20: {full_kr20:.3f}"
)

print(
    "All Supplementary Table S8 validation checks passed."
)

In [ ]:
# ==================================================
# SAVE OUTPUTS
# ==================================================

supplementary_table_s8.to_csv(
    "outputs/tables/supplementary/supplementary_table_s8_internal_consistency_of_8_domain_ace_score.csv",
    index=False
)

print(
    "Saved: "
    "outputs/tables/supplementary/supplementary_table_s8_internal_consistency_of_8_domain_ace_score.csv"
)

## Clinical and social determinant outcomes

Construct outcome flags and the complete analysis dataset.


In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings


# ==================================================
# CLINICAL OUTCOMES FOR ALL EHHWB PARTICIPANTS
# ==================================================

ehhwb_survey_concept_id = 1703970

clinical_outcome_flags = client.query(
    f"""
    WITH earliest_ehhwb AS (
        SELECT
            person_id
        FROM `{cdr}.survey_conduct`
        WHERE survey_source_concept_id = @ehhwb_survey_concept_id
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY person_id
            ORDER BY
                survey_start_datetime ASC,
                survey_conduct_id ASC
        ) = 1
    ),

    icd10_phenotype_concepts AS (
        SELECT
            CASE
                WHEN REGEXP_CONTAINS(
                    concept_code,
                    r'^(F32|F33)'
                )
                    THEN 'mdd'

                WHEN REGEXP_CONTAINS(
                    concept_code,
                    r'^F43\\.1'
                )
                    THEN 'ptsd'

                WHEN REGEXP_CONTAINS(
                    concept_code,
                    r'^F31'
                )
                    THEN 'bipolar'

                WHEN REGEXP_CONTAINS(
                    concept_code,
                    r'^F41\\.1'
                )
                    THEN 'gad'

                WHEN (
                    REGEXP_CONTAINS(
                        concept_code,
                        r'^R45\\.851'
                    )
                    OR REGEXP_CONTAINS(
                        concept_code,
                        r'^(X7[1-9]|X8[0-3]|T14\\.91)'
                    )
                    OR (
                        REGEXP_CONTAINS(
                            concept_code,
                            r'^(T3[6-9]|T4[0-9]|T5[0-9]|T6[0-5]|T71)'
                        )
                        AND LOWER(concept_name)
                            LIKE '%intentional self-harm%'
                    )
                )
                    THEN 'suicidal_ideation_or_self_harm'

                WHEN REGEXP_CONTAINS(
                    concept_code,
                    r'^(F10|F11|F1[2-6]|F18|F19)'
                )
                AND concept_code NOT IN (
                    'F10.9', 'F10.90',
                    'F11.9', 'F11.90',
                    'F12.9', 'F12.90',
                    'F13.9', 'F13.90',
                    'F14.9', 'F14.90',
                    'F15.9', 'F15.90',
                    'F16.9', 'F16.90',
                    'F18.9', 'F18.90',
                    'F19.9', 'F19.90'
                )
                    THEN 'substance_use_disorder'

                WHEN REGEXP_CONTAINS(
                    concept_code,
                    r'^G89\\.2'
                )
                    THEN 'chronic_pain'

                WHEN REGEXP_CONTAINS(
                    concept_code,
                    r'^(G47|F51)'
                )
                    THEN 'sleep_disorder'

                WHEN REGEXP_CONTAINS(
                    concept_code,
                    r'^E66'
                )
                    THEN 'obesity'
            END AS phenotype,

            concept_id

        FROM `{cdr}.concept`

        WHERE vocabulary_id = 'ICD10CM'
          AND invalid_reason IS NULL
          AND (
                REGEXP_CONTAINS(
                    concept_code,
                    r'^(F32|F33)'
                )
                OR REGEXP_CONTAINS(
                    concept_code,
                    r'^F43\\.1'
                )
                OR REGEXP_CONTAINS(
                    concept_code,
                    r'^F31'
                )
                OR REGEXP_CONTAINS(
                    concept_code,
                    r'^F41\\.1'
                )
                OR REGEXP_CONTAINS(
                    concept_code,
                    r'^R45\\.851'
                )
                OR REGEXP_CONTAINS(
                    concept_code,
                    r'^(X7[1-9]|X8[0-3]|T14\\.91)'
                )
                OR (
                    REGEXP_CONTAINS(
                        concept_code,
                        r'^(T3[6-9]|T4[0-9]|T5[0-9]|T6[0-5]|T71)'
                    )
                    AND LOWER(concept_name)
                        LIKE '%intentional self-harm%'
                )
                OR REGEXP_CONTAINS(
                    concept_code,
                    r'^(F10|F11|F1[2-6]|F18|F19)'
                )
                OR REGEXP_CONTAINS(
                    concept_code,
                    r'^G89\\.2'
                )
                OR REGEXP_CONTAINS(
                    concept_code,
                    r'^(G47|F51)'
                )
                OR REGEXP_CONTAINS(
                    concept_code,
                    r'^E66'
                )
          )
    ),

    condition_hits AS (
        SELECT DISTINCT
            co.person_id,
            ipc.phenotype
        FROM `{cdr}.condition_occurrence` co

        INNER JOIN earliest_ehhwb e
            ON co.person_id = e.person_id

        INNER JOIN icd10_phenotype_concepts ipc
            ON co.condition_source_concept_id
                = ipc.concept_id

        WHERE ipc.phenotype IS NOT NULL
    ),

    observation_hits AS (
        SELECT DISTINCT
            o.person_id,
            ipc.phenotype
        FROM `{cdr}.observation` o

        INNER JOIN earliest_ehhwb e
            ON o.person_id = e.person_id

        INNER JOIN icd10_phenotype_concepts ipc
            ON o.observation_source_concept_id
                = ipc.concept_id

        WHERE ipc.phenotype IS NOT NULL
    ),

    all_hits AS (
        SELECT * FROM condition_hits

        UNION DISTINCT

        SELECT * FROM observation_hits
    )

    SELECT
        e.person_id,

        MAX(IF(
            h.phenotype = 'mdd',
            1,
            0
        )) AS mdd,

        MAX(IF(
            h.phenotype = 'ptsd',
            1,
            0
        )) AS ptsd,

        MAX(IF(
            h.phenotype = 'bipolar',
            1,
            0
        )) AS bipolar,

        MAX(IF(
            h.phenotype = 'gad',
            1,
            0
        )) AS gad,

        MAX(IF(
            h.phenotype = 'suicidal_ideation_or_self_harm',
            1,
            0
        )) AS suicidal_ideation_or_self_harm,

        MAX(IF(
            h.phenotype = 'substance_use_disorder',
            1,
            0
        )) AS substance_use_disorder,

        MAX(IF(
            h.phenotype = 'chronic_pain',
            1,
            0
        )) AS chronic_pain,

        MAX(IF(
            h.phenotype = 'sleep_disorder',
            1,
            0
        )) AS sleep_disorder,

        MAX(IF(
            h.phenotype = 'obesity',
            1,
            0
        )) AS obesity

    FROM earliest_ehhwb e

    LEFT JOIN all_hits h
        ON e.person_id = h.person_id

    GROUP BY e.person_id

    ORDER BY e.person_id
    """,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter(
                "ehhwb_survey_concept_id",
                "INT64",
                ehhwb_survey_concept_id
            )
        ]
    ),
).to_dataframe()


assert clinical_outcome_flags["person_id"].is_unique

print(
    "EHHWB participants with clinical flags: "
    f"{len(clinical_outcome_flags):,}"
)

display(clinical_outcome_flags.head())

In [ ]:
# ==================================================
# SDOH OUTCOMES
# ==================================================

food_qid = 40192517
isolation_qid = 40192501
neighborhood_qid = 40192384

food_positive_ids = [
    45877955,  # Sometimes true
    36309834   # Often true
]

isolation_positive_ids = [
    45882528,  # Sometimes
    45884455   # Often
]

neighborhood_positive_ids = [
    45884599,  # Disagree
    45882953   # Strongly disagree
]

skip_id = 903096


sdoh_responses = client.query(
    f"""
    WITH raw_responses AS (
        SELECT
            o.person_id,
            o.observation_source_concept_id
                AS question_id,
            o.value_as_concept_id,
            o.observation_datetime,
            o.observation_id

        FROM `{cdr}.observation` o

        WHERE o.observation_source_concept_id IN (
            @food_qid,
            @isolation_qid,
            @neighborhood_qid
        )

          AND o.value_as_concept_id IS NOT NULL
          AND o.value_as_concept_id != @skip_id
    )

    SELECT
        person_id,
        question_id,
        value_as_concept_id

    FROM raw_responses

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY
            person_id,
            question_id
        ORDER BY
            observation_datetime ASC,
            observation_id ASC
    ) = 1
    """,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter(
                "food_qid",
                "INT64",
                food_qid
            ),
            bigquery.ScalarQueryParameter(
                "isolation_qid",
                "INT64",
                isolation_qid
            ),
            bigquery.ScalarQueryParameter(
                "neighborhood_qid",
                "INT64",
                neighborhood_qid
            ),
            bigquery.ScalarQueryParameter(
                "skip_id",
                "INT64",
                skip_id
            ),
        ]
    ),
).to_dataframe()


assert not sdoh_responses.duplicated(
    ["person_id", "question_id"]
).any()


# --------------------------------------------------
# Convert responses to binary outcomes
# --------------------------------------------------

food_insecurity_df = (
    sdoh_responses.loc[
        sdoh_responses["question_id"].eq(food_qid),
        [
            "person_id",
            "value_as_concept_id"
        ]
    ]
    .copy()
)

food_insecurity_df["food_insecurity"] = (
    food_insecurity_df["value_as_concept_id"]
    .isin(food_positive_ids)
    .astype(int)
)

food_insecurity_df = food_insecurity_df[
    [
        "person_id",
        "food_insecurity"
    ]
]


social_isolation_df = (
    sdoh_responses.loc[
        sdoh_responses["question_id"].eq(isolation_qid),
        [
            "person_id",
            "value_as_concept_id"
        ]
    ]
    .copy()
)

social_isolation_df["social_isolation"] = (
    social_isolation_df["value_as_concept_id"]
    .isin(isolation_positive_ids)
    .astype(int)
)

social_isolation_df = social_isolation_df[
    [
        "person_id",
        "social_isolation"
    ]
]


unsafe_neighborhood_df = (
    sdoh_responses.loc[
        sdoh_responses["question_id"].eq(neighborhood_qid),
        [
            "person_id",
            "value_as_concept_id"
        ]
    ]
    .copy()
)

unsafe_neighborhood_df["unsafe_neighborhood"] = (
    unsafe_neighborhood_df["value_as_concept_id"]
    .isin(neighborhood_positive_ids)
    .astype(int)
)

unsafe_neighborhood_df = unsafe_neighborhood_df[
    [
        "person_id",
        "unsafe_neighborhood"
    ]
]


print(
    f"Food insecurity valid responses: "
    f"{len(food_insecurity_df):,}"
)

print(
    f"Social isolation valid responses: "
    f"{len(social_isolation_df):,}"
)

print(
    f"Unsafe neighborhood valid responses: "
    f"{len(unsafe_neighborhood_df):,}"
)

In [ ]:
# ==================================================
# CREATE PERSON-LEVEL OUTCOME DATASET
# ==================================================

analysis_df = (
    complete_ace_with_demographics[
        [
            "person_id",
            "ace_burden_score",
            "age_group",
            "gender",
            "race",
            "ethnicity"
        ]
    ]
    .merge(
        clinical_outcome_flags,
        on="person_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        food_insecurity_df,
        on="person_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        social_isolation_df,
        on="person_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        unsafe_neighborhood_df,
        on="person_id",
        how="left",
        validate="one_to_one"
    )
)


# Clinical absence means no qualifying diagnosis was found
clinical_columns = [
    "ptsd",
    "bipolar",
    "suicidal_ideation_or_self_harm",
    "substance_use_disorder",
    "mdd",
    "gad",
    "sleep_disorder",
    "obesity",
    "chronic_pain"
]

analysis_df[clinical_columns] = (
    analysis_df[clinical_columns]
    .fillna(0)
    .astype(int)
)


# --------------------------------------------------
# Standardize age categories
# --------------------------------------------------

analysis_df["age_group"] = (
    analysis_df["age_group"]
    .replace({
        "<=30": "18-30",
        "≤30": "18-30"
    })
)


valid_age_groups = [
    "18-30",
    "31-45",
    "46-60",
    "61-75",
    "76+"
]

analysis_df.loc[
    ~analysis_df["age_group"].isin(valid_age_groups),
    "age_group"
] = "Missing"


# --------------------------------------------------
# Make missing demographics explicit
# --------------------------------------------------

for column in [
    "gender",
    "race",
    "ethnicity"
]:
    analysis_df[column] = (
        analysis_df[column]
        .astype("object")
        .where(
            analysis_df[column].notna(),
            "Missing"
        )
        .astype(str)
    )


analysis_df["age_group"] = (
    analysis_df["age_group"]
    .astype(str)
)


assert analysis_df["person_id"].is_unique

print(
    "Complete 8-domain analysis cohort: "
    f"{len(analysis_df):,}"
)

display(analysis_df.head())

In [ ]:
# ==================================================
# OUTCOME DEFINITIONS
# ==================================================

outcome_metadata = {
    "ptsd": {
        "domain": "Clinical",
        "label": "PTSD"
    },

    "food_insecurity": {
        "domain": "SDOH",
        "label": "Food insecurity"
    },

    "bipolar": {
        "domain": "Clinical",
        "label": "Bipolar disorder"
    },

    "suicidal_ideation_or_self_harm": {
        "domain": "Clinical",
        "label": "Suicidal ideation/self-harm"
    },

    "substance_use_disorder": {
        "domain": "Clinical",
        "label": "Substance use disorder"
    },

    "social_isolation": {
        "domain": "SDOH",
        "label": "Social isolation"
    },

    "unsafe_neighborhood": {
        "domain": "SDOH",
        "label": "Unsafe neighborhood"
    },

    "mdd": {
        "domain": "Clinical",
        "label": "Major depressive disorder"
    },

    "gad": {
        "domain": "Clinical",
        "label": "Generalized anxiety disorder"
    },

    "sleep_disorder": {
        "domain": "Clinical",
        "label": "Sleep disorder"
    },

    "obesity": {
        "domain": "Clinical",
        "label": "Obesity"
    },

    "chronic_pain": {
        "domain": "Clinical",
        "label": "Chronic pain"
    }
}

In [ ]:
# ==================================================
# MODEL FUNCTIONS
# ==================================================

def format_p_value(p_value):

    if pd.isna(p_value) or not np.isfinite(p_value):
        return "NE"

    if p_value < 0.001:
        return "<0.001"

    return f"{p_value:.4f}".rstrip("0").rstrip(".")


def format_or_ci(
    odds_ratio,
    ci_lower,
    ci_upper
):

    if not all(
        np.isfinite(value)
        for value in [
            odds_ratio,
            ci_lower,
            ci_upper
        ]
    ):
        return "NE"

    return (
        f"{odds_ratio:.2f} "
        f"({ci_lower:.2f}-{ci_upper:.2f})"
    )


def fit_continuous_ace_model(
    df,
    outcome,
    adjusted=True
):

    required_columns = [
        outcome,
        "ace_burden_score"
    ]

    if adjusted:
        required_columns += [
            "age_group",
            "gender",
            "race",
            "ethnicity"
        ]

    model_df = (
        df[required_columns]
        .dropna(subset=[outcome])
        .copy()
    )

    model_df[outcome] = (
        model_df[outcome]
        .astype(int)
    )

    model_n = len(model_df)
    cases_n = int(model_df[outcome].sum())
    prevalence = (
        100 * cases_n / model_n
        if model_n > 0
        else np.nan
    )

    if cases_n == 0 or cases_n == model_n:
        return {
            "N": model_n,
            "Cases": cases_n,
            "Prevalence, %": prevalence,
            "OR": np.nan,
            "CI Lower": np.nan,
            "CI Upper": np.nan,
            "p-value raw": np.nan
        }

    if adjusted:

        formula = f"""
            {outcome}
            ~ ace_burden_score
            + C(
                age_group,
                Treatment(reference="61-75")
              )
            + C(
                gender,
                Treatment(reference="Female")
              )
            + C(
                race,
                Treatment(reference="White")
              )
            + C(
                ethnicity,
                Treatment(
                    reference="Not Hispanic or Latino"
                )
              )
        """

    else:

        formula = f"""
            {outcome}
            ~ ace_burden_score
        """

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        result = smf.glm(
            formula=formula,
            data=model_df,
            family=sm.families.Binomial()
        ).fit(
            maxiter=1000
        )

    beta = result.params["ace_burden_score"]
    p_value = result.pvalues["ace_burden_score"]

    confidence_interval = result.conf_int().loc[
        "ace_burden_score"
    ]

    odds_ratio = np.exp(beta)
    ci_lower = np.exp(confidence_interval.iloc[0])
    ci_upper = np.exp(confidence_interval.iloc[1])

    return {
        "N": model_n,
        "Cases": cases_n,
        "Prevalence, %": prevalence,
        "OR": odds_ratio,
        "CI Lower": ci_lower,
        "CI Upper": ci_upper,
        "p-value raw": p_value
    }

## Supplementary Tables S9 and S10

Adjusted and unadjusted associations between continuous eight-domain ACE burden and predefined outcomes.


In [ ]:
# ==================================================
# SUPPLEMENTARY TABLE S9:
# ADJUSTED ASSOCIATIONS
# ==================================================

s9_rows = []

for outcome, metadata in outcome_metadata.items():

    model_stats = fit_continuous_ace_model(
        df=analysis_df,
        outcome=outcome,
        adjusted=True
    )

    s9_rows.append({
        "Outcome Domain": metadata["domain"],
        "Outcome": metadata["label"],
        "N": model_stats["N"],
        "Cases": model_stats["Cases"],
        "Prevalence, %": model_stats[
            "Prevalence, %"
        ],
        "Adjusted OR (95% CI)": format_or_ci(
            model_stats["OR"],
            model_stats["CI Lower"],
            model_stats["CI Upper"]
        ),
        "p-value": format_p_value(
            model_stats["p-value raw"]
        ),
        "_or_sort": model_stats["OR"]
    })


supplementary_table_s9 = pd.DataFrame(
    s9_rows
)


# Sort from strongest positive association downward
supplementary_table_s9 = (
    supplementary_table_s9
    .sort_values(
        "_or_sort",
        ascending=False,
        na_position="last"
    )
    .drop(columns="_or_sort")
    .reset_index(drop=True)
)


supplementary_table_s9[
    "Prevalence, %"
] = (
    supplementary_table_s9[
        "Prevalence, %"
    ]
    .round(1)
)


supplementary_table_s9 = supplementary_table_s9[
    [
        "Outcome Domain",
        "Outcome",
        "N",
        "Cases",
        "Prevalence, %",
        "Adjusted OR (95% CI)",
        "p-value"
    ]
]


display(supplementary_table_s9)

In [ ]:
# ==================================================
# SUPPLEMENTARY TABLE S10:
# UNADJUSTED ASSOCIATIONS
# ==================================================

s10_rows = []

for outcome, metadata in outcome_metadata.items():

    model_stats = fit_continuous_ace_model(
        df=analysis_df,
        outcome=outcome,
        adjusted=False
    )

    s10_rows.append({
        "Outcome Domain": metadata["domain"],
        "Outcome": metadata["label"],
        "N": model_stats["N"],
        "Cases": model_stats["Cases"],
        "Prevalence, %": model_stats[
            "Prevalence, %"
        ],
        "Unadjusted OR (95% CI)": format_or_ci(
            model_stats["OR"],
            model_stats["CI Lower"],
            model_stats["CI Upper"]
        ),
        "p-value": format_p_value(
            model_stats["p-value raw"]
        ),
        "_or_sort": model_stats["OR"]
    })


supplementary_table_s10 = pd.DataFrame(
    s10_rows
)


supplementary_table_s10 = (
    supplementary_table_s10
    .sort_values(
        "_or_sort",
        ascending=False,
        na_position="last"
    )
    .drop(columns="_or_sort")
    .reset_index(drop=True)
)


supplementary_table_s10[
    "Prevalence, %"
] = (
    supplementary_table_s10[
        "Prevalence, %"
    ]
    .round(1)
)


supplementary_table_s10 = supplementary_table_s10[
    [
        "Outcome Domain",
        "Outcome",
        "N",
        "Cases",
        "Prevalence, %",
        "Unadjusted OR (95% CI)",
        "p-value"
    ]
]


display(supplementary_table_s10)

In [ ]:
# ==================================================
# VALIDATION
# ==================================================

clinical_n = len(analysis_df)

for outcome in clinical_columns:

    table_n = supplementary_table_s9.loc[
        supplementary_table_s9[
            "Outcome"
        ].eq(outcome_metadata[outcome]["label"]),
        "N"
    ].iloc[0]

    assert table_n == clinical_n, (
        f"{outcome}: clinical denominator is "
        f"{table_n:,}, expected {clinical_n:,}."
    )


sdoh_expected_n = {
    "food_insecurity": analysis_df[
        "food_insecurity"
    ].notna().sum(),

    "social_isolation": analysis_df[
        "social_isolation"
    ].notna().sum(),

    "unsafe_neighborhood": analysis_df[
        "unsafe_neighborhood"
    ].notna().sum()
}


for outcome, expected_n in sdoh_expected_n.items():

    outcome_label = outcome_metadata[outcome][
        "label"
    ]

    adjusted_n = supplementary_table_s9.loc[
        supplementary_table_s9[
            "Outcome"
        ].eq(outcome_label),
        "N"
    ].iloc[0]

    unadjusted_n = supplementary_table_s10.loc[
        supplementary_table_s10[
            "Outcome"
        ].eq(outcome_label),
        "N"
    ].iloc[0]

    assert adjusted_n == expected_n
    assert unadjusted_n == expected_n


assert supplementary_table_s9.shape[0] == 12
assert supplementary_table_s10.shape[0] == 12

print("Clinical analysis N:", f"{clinical_n:,}")

for outcome, expected_n in sdoh_expected_n.items():
    print(
        f"{outcome}: N = {expected_n:,}"
    )

print("All S9 and S10 validation checks passed.")

In [ ]:
# ==================================================
# SAVE OUTPUTS
# ==================================================

supplementary_table_s9.to_csv(
    "outputs/tables/supplementary/supplementary_table_s9_adjusted_associations_between_continuous_8_domain_ace_burden_and_outcomes.csv",
    index=False
)

supplementary_table_s10.to_csv(
    "outputs/tables/supplementary/supplementary_table_s10_unadjusted_associations_between_continuous_8_domain_ace_burden_and_outcomes.csv",
    index=False
)

print(
    "Saved: "
    "outputs/tables/supplementary/supplementary_table_s9_adjusted_associations_between_continuous_8_domain_ace_burden_and_outcomes.csv"
)

print(
    "Saved: "
    "outputs/tables/supplementary/supplementary_table_s10_unadjusted_associations_between_continuous_8_domain_ace_burden_and_outcomes.csv"
)

## Main Figure 4

Associations between eight-domain ACE burden score and predefined clinical and social determinant outcomes.

**Required input:** Supplementary Table S9. Run that section first if its exported CSV file is not present.


In [ ]:
# -----------------------------
# Figure 4: Associations between eight-domain ACE burden score and outcomes
# -----------------------------

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Load results
# -----------------------------

forest_df = pd.read_csv(
    "outputs/tables/supplementary/supplementary_table_s9_adjusted_associations_between_continuous_8_domain_ace_burden_and_outcomes.csv"
)

display(forest_df)


# -----------------------------
# Extract adjusted OR and 95% CI
# -----------------------------

or_ci = forest_df["Adjusted OR (95% CI)"].str.extract(
    r"(?P<or>\d+(?:\.\d+)?)\s*"
    r"\(\s*(?P<ci_lower>\d+(?:\.\d+)?)\s*-\s*"
    r"(?P<ci_upper>\d+(?:\.\d+)?)\s*\)"
)

forest_df[["or", "ci_lower", "ci_upper"]] = or_ci.astype(float)


# -----------------------------
# Order outcomes
# -----------------------------
# The CSV is already ordered from strongest to weakest association.
# Reverse the rows so the strongest association appears at the top
# after plotting from bottom to top.

forest_df = forest_df.iloc[::-1].reset_index(drop=True)

display(forest_df)


# -----------------------------
# Save combined figure data
# -----------------------------

forest_df.to_csv(
    "outputs/figure_data/main/figure_4_associations_between_8_domain_ace_burden_and_outcomes_data.csv",
    index=False
)


# -----------------------------
# Forest plot
# -----------------------------

fig, ax = plt.subplots(figsize=(9, 6.5))

y = np.arange(len(forest_df))

ax.errorbar(
    forest_df["or"],
    y,
    xerr=[
        forest_df["or"] - forest_df["ci_lower"],
        forest_df["ci_upper"] - forest_df["or"]
    ],
    fmt="o",
    capsize=3
)

ax.axvline(
    1,
    linestyle="--",
    linewidth=1
)

ax.set_yticks(y)
ax.set_yticklabels(forest_df["Outcome"])

ax.set_xlabel(
    "Adjusted Odds Ratio per 1-Point Increase in ACE Burden"
)

ax.set_ylabel("")

ax.set_title(
    "Associations Between ACE Burden and Clinical and Social Outcomes"
)


# -----------------------------
# Add OR and 95% CI labels
# -----------------------------

for i, row in forest_df.iterrows():
    ax.text(
        row["ci_upper"] + 0.012,
        i,
        (
            f'{row["or"]:.2f} '
            f'({row["ci_lower"]:.2f}–{row["ci_upper"]:.2f})'
        ),
        va="center",
        fontsize=9
    )


# -----------------------------
# Axis limits and formatting
# -----------------------------

ax.set_xlim(
    0.98,
    forest_df["ci_upper"].max() + 0.24
)

fig.tight_layout()

plt.show()




In [ ]:
# ==================================================
# SAVE FIGURE
# ==================================================

fig.savefig(
    "outputs/figures/main/figure_4_associations_between_8_domain_ace_burden_and_outcomes.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    "outputs/figures/main/figure_4_associations_between_8_domain_ace_burden_and_outcomes.pdf",
    bbox_inches="tight"
)


## Supplementary Table S11

Prevalence of outcomes by eight-domain ACE burden score.


In [ ]:
import pandas as pd
import numpy as np


# ==================================================
# SUPPLEMENTARY TABLE S11:
# PREVALENCE OF OUTCOMES BY 8-DOMAIN ACE BURDEN SCORE
# ==================================================

outcome_metadata = {
    "bipolar": {
        "domain": "Clinical",
        "label": "Bipolar disorder"
    },

    "chronic_pain": {
        "domain": "Clinical",
        "label": "Chronic pain"
    },

    "gad": {
        "domain": "Clinical",
        "label": "Generalized anxiety disorder"
    },

    "mdd": {
        "domain": "Clinical",
        "label": "Major depressive disorder"
    },

    "obesity": {
        "domain": "Clinical",
        "label": "Obesity"
    },

    "ptsd": {
        "domain": "Clinical",
        "label": "PTSD"
    },

    "sleep_disorder": {
        "domain": "Clinical",
        "label": "Sleep disorder"
    },

    "substance_use_disorder": {
        "domain": "Clinical",
        "label": "Substance use disorder"
    },

    "suicidal_ideation_or_self_harm": {
        "domain": "Clinical",
        "label": "Suicidal ideation/self-harm"
    },

    "food_insecurity": {
        "domain": "SDOH",
        "label": "Food insecurity"
    },

    "social_isolation": {
        "domain": "SDOH",
        "label": "Social isolation"
    },

    "unsafe_neighborhood": {
        "domain": "SDOH",
        "label": "Unsafe neighborhood"
    }
}


# --------------------------------------------------
# Prepare analysis data
# --------------------------------------------------

s11_analysis_df = analysis_df.copy()

s11_analysis_df = s11_analysis_df.loc[
    s11_analysis_df["ace_burden_score"].notna()
].copy()

s11_analysis_df["ace_burden_score"] = (
    s11_analysis_df["ace_burden_score"]
    .astype(int)
)


# Confirm new ACE score ranges from 0 to 8
assert s11_analysis_df[
    "ace_burden_score"
].between(0, 8).all(), (
    "ACE burden scores outside the expected 0-8 range were found."
)

print(
    "Complete 8-domain ACE cohort: "
    f"{s11_analysis_df['person_id'].nunique():,}"
)

print(
    "Observed ACE burden scores:",
    sorted(
        s11_analysis_df[
            "ace_burden_score"
        ].unique()
    )
)

In [ ]:
# ==================================================
# BUILD SUPPLEMENTARY TABLE S11
# ==================================================

s11_rows = []

for outcome, metadata in outcome_metadata.items():

    # --------------------------------------------------
    # Outcome-specific analytic cohort
    #
    # Clinical outcomes should be complete 0/1 flags.
    # SDOH outcomes retain missing values for participants
    # without a valid response.
    # --------------------------------------------------

    outcome_df = (
        s11_analysis_df[
            [
                "person_id",
                "ace_burden_score",
                outcome
            ]
        ]
        .dropna(subset=[outcome])
        .copy()
    )

    outcome_df[outcome] = (
        outcome_df[outcome]
        .astype(int)
    )

    # --------------------------------------------------
    # Summarize each ACE burden score from 0 to 8
    # --------------------------------------------------

    grouped = (
        outcome_df
        .groupby(
            "ace_burden_score",
            as_index=False
        )
        .agg(
            people_n=(
                "person_id",
                "nunique"
            ),
            cases_n=(
                outcome,
                "sum"
            )
        )
    )

    # Include all scores 0-8 even if one score has no people
    grouped = (
        grouped
        .set_index("ace_burden_score")
        .reindex(range(0, 9), fill_value=0)
        .rename_axis("ace_burden_score")
        .reset_index()
    )

    grouped["prevalence_pct"] = np.where(
        grouped["people_n"] > 0,
        100 * grouped["cases_n"] / grouped["people_n"],
        np.nan
    )

    for _, row in grouped.iterrows():

        s11_rows.append({
            "Outcome Domain": metadata["domain"],
            "Outcome": metadata["label"],
            "ACE Burden Score": int(
                row["ace_burden_score"]
            ),
            "People, n": int(
                row["people_n"]
            ),
            "Cases, n": int(
                row["cases_n"]
            ),
            "Prevalence, %": row[
                "prevalence_pct"
            ]
        })


supplementary_table_s11 = pd.DataFrame(
    s11_rows
)


# --------------------------------------------------
# Round prevalence
# --------------------------------------------------

supplementary_table_s11[
    "Prevalence, %"
] = (
    supplementary_table_s11[
        "Prevalence, %"
    ]
    .round(2)
)


# --------------------------------------------------
# Final column order
# --------------------------------------------------

supplementary_table_s11 = supplementary_table_s11[
    [
        "Outcome Domain",
        "Outcome",
        "ACE Burden Score",
        "People, n",
        "Cases, n",
        "Prevalence, %"
    ]
]


display(supplementary_table_s11)

In [ ]:
# ==================================================
# VALIDATION
# ==================================================

expected_scores = list(range(0, 9))

for outcome, metadata in outcome_metadata.items():

    outcome_label = metadata["label"]

    outcome_rows = supplementary_table_s11.loc[
        supplementary_table_s11[
            "Outcome"
        ].eq(outcome_label)
    ]

    assert outcome_rows[
        "ACE Burden Score"
    ].tolist() == expected_scores, (
        f"{outcome_label}: ACE scores are not exactly 0-8."
    )

    expected_n = int(
        s11_analysis_df[outcome]
        .notna()
        .sum()
    )

    observed_n = int(
        outcome_rows["People, n"].sum()
    )

    assert observed_n == expected_n, (
        f"{outcome_label}: People counts sum to "
        f"{observed_n:,}, expected {expected_n:,}."
    )

    expected_cases = int(
        s11_analysis_df[outcome]
        .fillna(0)
        .sum()
    )

    observed_cases = int(
        outcome_rows["Cases, n"].sum()
    )

    assert observed_cases == expected_cases, (
        f"{outcome_label}: Case counts sum to "
        f"{observed_cases:,}, expected {expected_cases:,}."
    )

    assert (
        outcome_rows["Cases, n"]
        <= outcome_rows["People, n"]
    ).all(), (
        f"{outcome_label}: cases exceed people "
        f"for at least one ACE score."
    )

    print(
        f"{outcome_label}: "
        f"N = {observed_n:,}; "
        f"cases = {observed_cases:,}"
    )


assert len(supplementary_table_s11) == (
    len(outcome_metadata) * 9
)

print(
    "All Supplementary Table S11 validation checks passed."
)

In [ ]:
# ==================================================
# SAVE OUTPUTS
# ==================================================

supplementary_table_s11.to_csv(
    "outputs/tables/supplementary/supplementary_table_s11_prevalence_of_outcomes_by_8_domain_ace_burden_score.csv",
    index=False
)

print(
    "Saved: "
    "outputs/tables/supplementary/supplementary_table_s11_prevalence_of_outcomes_by_8_domain_ace_burden_score.csv"
)

## Main Figure 5 and Supplementary Figure S2

Dose-response relationships between eight-domain ACE burden score and predefined clinical and social determinant outcomes. The main and supplementary figures are generated together from Supplementary Table S11.

**Required input:** Supplementary Table S11. Run that section first if its exported CSV file is not present.


In [ ]:
# -----------------------------
# Figure 5 and Supplementary Figure S2: Dose-response plots
# One 2x2 main figure and one 3-column by 3-row supplementary figure
# -----------------------------

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Load plotting data
# -----------------------------

dose_response = pd.read_csv(
    "outputs/tables/supplementary/supplementary_table_s11_prevalence_of_outcomes_by_8_domain_ace_burden_score.csv"
)

dose_response = dose_response.rename(columns={
    "Outcome Domain": "outcome_domain",
    "Outcome": "outcome",
    "ACE Burden Score": "ace_score",
    "People, n": "n_total",
    "Cases, n": "n_cases",
    "Prevalence, %": "prevalence_percent"
})

dose_response["ace_score"] = pd.to_numeric(
    dose_response["ace_score"]
).astype(int)

dose_response["n_total"] = pd.to_numeric(
    dose_response["n_total"]
).astype(int)

dose_response["n_cases"] = pd.to_numeric(
    dose_response["n_cases"]
).astype(int)

dose_response["prevalence_percent"] = pd.to_numeric(
    dose_response["prevalence_percent"]
)

display(dose_response)


# -----------------------------
# Outcome groupings
# -----------------------------

# The selected strongest outcomes form Main Figure 5.
# The eight additional outcomes form one 3-column by 3-row
# Supplementary Figure S2.
figure_groups = {
    "outputs/figures/main/figure_5_dose_response_relationships": {
        "outcomes": [
            "PTSD",
            "Food insecurity",
            "Bipolar disorder",
            "Suicidal ideation/self-harm"
        ],
        "nrows": 2,
        "ncols": 2,
        "figsize": (12, 10)
    },

    "outputs/figures/supplementary/supplementary_figure_s2_additional_outcomes": {
        "outcomes": [
            "Substance use disorder",
            "Social isolation",
            "Unsafe neighborhood",
            "Major depressive disorder",
            "Generalized anxiety disorder",
            "Sleep disorder",
            "Obesity",
            "Chronic pain"
        ],
        "nrows": 3,
        "ncols": 3,
        "figsize": (18, 15)
    }
}


# -----------------------------
# Panel plotting function
# -----------------------------

def plot_outcome_panel(ax, outcome_label):

    temp = dose_response[
        dose_response["outcome"].eq(outcome_label)
    ].copy()

    temp = temp.sort_values(
        "ace_score"
    )

    ax.plot(
        temp["ace_score"],
        temp["prevalence_percent"],
        marker="o",
        linewidth=2
    )

    ax.set_xticks(
        np.arange(0, 9)
    )

    ax.set_xticklabels(
        [str(score) for score in range(9)]
    )

    ax.set_xlabel(
        "ACE Burden Score",
        labelpad=52
    )

    ax.set_ylabel(
        "Outcome Prevalence (%)"
    )

    ax.set_title(
        outcome_label
    )


    # -----------------------------
    # Participant counts
    # -----------------------------

    for _, row in temp.iterrows():

        ax.text(
            row["ace_score"],
            -0.17,
            f'n={int(row["n_total"]):,}',
            ha="center",
            va="top",
            fontsize=7,
            rotation=45,
            rotation_mode="anchor",
            transform=ax.get_xaxis_transform(),
            clip_on=False
        )


    # -----------------------------
    # Outcome-specific y-axis scale
    # -----------------------------

    y_min = temp["prevalence_percent"].min()
    y_max = temp["prevalence_percent"].max()
    y_range = y_max - y_min

    padding = max(
        y_range * 0.15,
        0.5
    )

    lower = max(
        0,
        y_min - padding
    )

    upper = y_max + padding

    ax.set_ylim(
        lower,
        upper
    )


# -----------------------------
# Generate Main Figure 5 and Supplementary Figure S2
# -----------------------------

for filename_prefix, figure_settings in figure_groups.items():

    outcomes = figure_settings["outcomes"]

    fig, axes = plt.subplots(
        nrows=figure_settings["nrows"],
        ncols=figure_settings["ncols"],
        figsize=figure_settings["figsize"]
    )

    axes = axes.flatten()

    panel_letters = [
        chr(ord("a") + index)
        for index in range(len(outcomes))
    ]

    for ax, panel_letter, outcome_label in zip(
        axes,
        panel_letters,
        outcomes
    ):

        plot_outcome_panel(
            ax=ax,
            outcome_label=outcome_label
        )

        ax.text(
            -0.12,
            1.08,
            panel_letter,
            transform=ax.transAxes,
            fontsize=14,
            fontweight="bold",
            va="top"
        )

    for unused_ax in axes[len(outcomes):]:
        unused_ax.axis("off")

    is_supplementary_figure_s2 = (
        "supplementary_figure_s2" in filename_prefix
    )

    fig.subplots_adjust(
        left=0.10,
        right=0.98,
        top=0.96 if is_supplementary_figure_s2 else 0.94,
        bottom=0.07 if is_supplementary_figure_s2 else 0.13,
        hspace=0.75,
        wspace=0.30
    )


    # ==================================================
    # SAVE FIGURES
    # ==================================================

    fig.savefig(
        f"{filename_prefix}.png",
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        f"{filename_prefix}.pdf",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)





## Supplementary Table S12

Continuous ACE score associations with predefined outcomes.


In [ ]:
# ==================================================
# PREPARE ANALYSIS DATA
# ==================================================

s12_analysis_df = analysis_df[
    [
        "person_id",
        "ace_burden_score",
        "age_group",
        "gender",
        "race",
        "ethnicity",
        *outcome_metadata.keys()
    ]
].copy()


# Keep only complete 8-domain ACE scores
s12_analysis_df = s12_analysis_df.loc[
    s12_analysis_df["ace_burden_score"].notna()
].copy()

s12_analysis_df["ace_burden_score"] = (
    s12_analysis_df["ace_burden_score"]
    .astype(int)
)

assert s12_analysis_df[
    "ace_burden_score"
].between(0, 8).all()


# Standardize age groups
s12_analysis_df["age_group"] = (
    s12_analysis_df["age_group"]
    .replace({
        "<=30": "18-30",
        "≤30": "18-30"
    })
)

valid_age_groups = [
    "18-30",
    "31-45",
    "46-60",
    "61-75",
    "76+"
]

s12_analysis_df.loc[
    ~s12_analysis_df["age_group"].isin(valid_age_groups),
    "age_group"
] = "Missing"


# Make missing demographics explicit
for column in [
    "age_group",
    "gender",
    "race",
    "ethnicity"
]:
    s12_analysis_df[column] = (
        s12_analysis_df[column]
        .astype("object")
        .where(
            s12_analysis_df[column].notna(),
            "Missing"
        )
        .astype(str)
    )


# Clinical outcomes should already be fully defined 0/1
clinical_outcomes = [
    outcome
    for outcome, metadata in outcome_metadata.items()
    if metadata["domain"] == "Clinical"
]

for outcome in clinical_outcomes:
    s12_analysis_df[outcome] = (
        s12_analysis_df[outcome]
        .fillna(0)
        .astype(int)
    )


# SDOH missingness is retained
sdoh_outcomes = [
    outcome
    for outcome, metadata in outcome_metadata.items()
    if metadata["domain"] == "SDOH"
]

for outcome in sdoh_outcomes:
    nonmissing_values = (
        s12_analysis_df.loc[
            s12_analysis_df[outcome].notna(),
            outcome
        ]
    )

    assert nonmissing_values.isin([0, 1]).all(), (
        f"{outcome} contains nonbinary values."
    )


assert s12_analysis_df["person_id"].is_unique

print(
    "Complete 8-domain ACE cohort: "
    f"{len(s12_analysis_df):,}"
)

for outcome in sdoh_outcomes:
    print(
        f"{outcome} valid responses: "
        f"{s12_analysis_df[outcome].notna().sum():,}"
    )

In [ ]:
# ==================================================
# HELPER FUNCTIONS
# ==================================================

def format_p_value(p_value):

    if pd.isna(p_value) or not np.isfinite(p_value):
        return "NE"

    if p_value < 0.001:
        return "<0.001"

    return f"{p_value:.4f}".rstrip("0").rstrip(".")


def format_or_ci(
    odds_ratio,
    ci_lower,
    ci_upper
):

    if not all(
        np.isfinite(value)
        for value in [
            odds_ratio,
            ci_lower,
            ci_upper
        ]
    ):
        return "NE"

    return (
        f"{odds_ratio:.2f} "
        f"({ci_lower:.2f}–{ci_upper:.2f})"
    )


def fit_binary_ace_model(
    df,
    outcome,
    threshold,
    adjusted
):

    exposure_column = f"ace_ge_{threshold}"

    model_df = df[
        [
            outcome,
            "ace_burden_score",
            "age_group",
            "gender",
            "race",
            "ethnicity"
        ]
    ].copy()

    model_df[exposure_column] = (
        model_df["ace_burden_score"]
        .ge(threshold)
        .astype(int)
    )

    required_columns = [
        outcome,
        exposure_column
    ]

    if adjusted:
        required_columns += [
            "age_group",
            "gender",
            "race",
            "ethnicity"
        ]

    # Outcome-specific complete cases
    model_df = (
        model_df[required_columns]
        .dropna()
        .copy()
    )

    model_df[outcome] = (
        model_df[outcome]
        .astype(int)
    )

    model_n = len(model_df)
    cases_n = int(model_df[outcome].sum())
    exposed_n = int(model_df[exposure_column].sum())

    exposed_pct = (
        100 * exposed_n / model_n
        if model_n > 0
        else np.nan
    )

    prevalence_pct = (
        100 * cases_n / model_n
        if model_n > 0
        else np.nan
    )

    if (
        model_n == 0
        or cases_n == 0
        or cases_n == model_n
    ):
        return {
            "N": model_n,
            "Cases": cases_n,
            "Exposed, n": exposed_n,
            "Exposed, %": exposed_pct,
            "Prevalence, %": prevalence_pct,
            "OR": np.nan,
            "CI Lower": np.nan,
            "CI Upper": np.nan,
            "p-value raw": np.nan
        }

    if adjusted:
        formula = f"""
            {outcome}
            ~ {exposure_column}
            + C(
                age_group,
                Treatment(reference="61-75")
              )
            + C(
                gender,
                Treatment(reference="Female")
              )
            + C(
                race,
                Treatment(reference="White")
              )
            + C(
                ethnicity,
                Treatment(
                    reference="Not Hispanic or Latino"
                )
              )
        """
    else:
        formula = f"""
            {outcome}
            ~ {exposure_column}
        """

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        result = smf.glm(
            formula=formula,
            data=model_df,
            family=sm.families.Binomial()
        ).fit(
            maxiter=1000
        )

    beta = result.params[exposure_column]
    p_value = result.pvalues[exposure_column]

    ci = result.conf_int().loc[
        exposure_column
    ]

    odds_ratio = np.exp(beta)
    ci_lower = np.exp(ci.iloc[0])
    ci_upper = np.exp(ci.iloc[1])

    return {
        "N": model_n,
        "Cases": cases_n,
        "Exposed, n": exposed_n,
        "Exposed, %": exposed_pct,
        "Prevalence, %": prevalence_pct,
        "OR": odds_ratio,
        "CI Lower": ci_lower,
        "CI Upper": ci_upper,
        "p-value raw": p_value
    }

In [ ]:
# ==================================================
# BUILD SUPPLEMENTARY TABLE S12
# CONTINUOUS ACE SCORE
# ==================================================

s12_rows = []

total_experiments = len(outcome_metadata) * 2
experiment_number = 0

for outcome, metadata in outcome_metadata.items():

    for adjusted in [False, True]:

        experiment_number += 1

        model_label = (
            "Adjusted"
            if adjusted
            else "Unadjusted"
        )

        model_stats = fit_continuous_ace_model(
            df=s12_analysis_df,
            outcome=outcome,
            adjusted=adjusted
        )

        result_row = {
            "Exposure": "Continuous ACE score",
            "Outcome Domain": metadata["domain"],
            "Outcome": metadata["label"],
            "Model": model_label,
            "N": model_stats["N"],
            "Cases": model_stats["Cases"],
            "Prevalence, %": round(
                model_stats["Prevalence, %"],
                1
            ),
            "OR (95% CI)": format_or_ci(
                model_stats["OR"],
                model_stats["CI Lower"],
                model_stats["CI Upper"]
            ),
            "p-value": format_p_value(
                model_stats["p-value raw"]
            )
        }

        s12_rows.append(result_row)

        print(
            f"[{experiment_number}/{total_experiments}] "
            f"Completed: Continuous ACE score | "
            f"{metadata['label']} | "
            f"{model_label}"
        )

        display(
            pd.DataFrame([result_row])
        )


supplementary_table_s12 = pd.DataFrame(
    s12_rows
)


supplementary_table_s12 = supplementary_table_s12[
    [
        "Exposure",
        "Outcome Domain",
        "Outcome",
        "Model",
        "N",
        "Cases",
        "Prevalence, %",
        "OR (95% CI)",
        "p-value"
    ]
]


print("\nAll continuous ACE score models completed.")

display(supplementary_table_s12)

In [ ]:
# ==================================================
# VALIDATION
# CONTINUOUS ACE SCORE
# ==================================================

expected_rows = (
    len(outcome_metadata)
    * 2
)

assert len(supplementary_table_s12) == expected_rows, (
    f"Expected {expected_rows} rows, "
    f"but found {len(supplementary_table_s12)}."
)


# Confirm there is only one exposure definition
assert supplementary_table_s12[
    "Exposure"
].eq(
    "Continuous ACE score"
).all(), (
    "Unexpected exposure label found."
)


for outcome, metadata in outcome_metadata.items():

    outcome_label = metadata["label"]

    # Rows eligible for this outcome-specific model
    outcome_complete = (
        s12_analysis_df.loc[
            s12_analysis_df[outcome].notna()
        ]
        .copy()
    )

    expected_n = len(outcome_complete)

    expected_cases = int(
        outcome_complete[outcome].sum()
    )

    expected_prevalence = round(
        100
        * expected_cases
        / expected_n,
        1
    )

    outcome_rows = supplementary_table_s12.loc[
        supplementary_table_s12[
            "Outcome"
        ].eq(outcome_label)
    ]

    assert len(outcome_rows) == 2, (
        f"{outcome_label}: expected 2 model rows, "
        f"but found {len(outcome_rows)}."
    )

    assert outcome_rows["N"].eq(
        expected_n
    ).all(), (
        f"{outcome_label}: incorrect N."
    )

    assert outcome_rows["Cases"].eq(
        expected_cases
    ).all(), (
        f"{outcome_label}: incorrect case count."
    )

    assert outcome_rows[
        "Prevalence, %"
    ].eq(
        expected_prevalence
    ).all(), (
        f"{outcome_label}: incorrect prevalence."
    )

    assert set(
        outcome_rows["Model"]
    ) == {
        "Unadjusted",
        "Adjusted"
    }, (
        f"{outcome_label}: missing or duplicated "
        f"model type."
    )


# Confirm exactly two models per outcome
model_counts = (
    supplementary_table_s12
    .groupby("Outcome")["Model"]
    .nunique()
)

assert model_counts.eq(2).all(), (
    "Each outcome must have exactly two model types."
)


# Confirm there are no duplicate model rows
duplicate_rows = supplementary_table_s12.duplicated(
    subset=[
        "Exposure",
        "Outcome",
        "Model"
    ]
)

assert not duplicate_rows.any(), (
    "Duplicate exposure-outcome-model rows found."
)


# Confirm model output fields are populated
for column in [
    "OR (95% CI)",
    "p-value"
]:
    assert supplementary_table_s12[
        column
    ].notna().all(), (
        f"Missing values found in {column}."
    )


print(
    "All continuous ACE Supplementary Table S12 "
    "validation checks passed."
)

In [ ]:
# ==================================================
# SAVE OUTPUTS
# ==================================================

supplementary_table_s12.to_csv(
    "outputs/tables/supplementary/supplementary_table_s12_continuous_ace_score_associations_with_outcomes.csv",
    index=False
)

print(
    "Saved: "
    "outputs/tables/supplementary/supplementary_table_s12_continuous_ace_score_associations_with_outcomes.csv"
)

## Restricted two-visit cohort

Construct the sensitivity-analysis cohort with at least two inpatient or outpatient visits.


In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings


# ==================================================
# PARTICIPANTS WITH AT LEAST 2 INPATIENT/OUTPATIENT
# VISITS
# ==================================================

two_visit_people = client.query(
    f"""
    SELECT
        person_id,
        COUNT(*) AS n_inpatient_outpatient_visits

    FROM `{cdr}.visit_occurrence`

    WHERE visit_concept_id IN (
        9201,  -- Inpatient Visit
        9202   -- Outpatient Visit
    )

    GROUP BY person_id

    HAVING COUNT(*) >= 2
    """
).to_dataframe()


assert two_visit_people["person_id"].is_unique

print(
    "Participants with at least 2 inpatient/outpatient visits: "
    f"{len(two_visit_people):,}"
)

In [ ]:
# ==================================================
# RESTRICT EXISTING 8-DOMAIN ANALYSIS DATASET
# ==================================================

analysis_df_2visits = (
    analysis_df
    .merge(
        two_visit_people,
        on="person_id",
        how="inner",
        validate="one_to_one"
    )
    .copy()
)


# Confirm complete 8-domain scores
analysis_df_2visits = analysis_df_2visits.loc[
    analysis_df_2visits[
        "ace_burden_score"
    ].notna()
].copy()

analysis_df_2visits["ace_burden_score"] = (
    analysis_df_2visits[
        "ace_burden_score"
    ].astype(int)
)


assert analysis_df_2visits[
    "person_id"
].is_unique

assert analysis_df_2visits[
    "ace_burden_score"
].between(0, 8).all()


print(
    "Complete 8-domain ACE participants with "
    "at least 2 visits: "
    f"{len(analysis_df_2visits):,}"
)

display(analysis_df_2visits.head())

In [ ]:
# ==================================================
# OUTCOME DEFINITIONS
# ==================================================

outcome_metadata_2visits = {
    "mdd": {
        "domain": "Clinical",
        "label": "MDD"
    },

    "ptsd": {
        "domain": "Clinical",
        "label": "PTSD"
    },

    "bipolar": {
        "domain": "Clinical",
        "label": "Bipolar disorder"
    },

    "gad": {
        "domain": "Clinical",
        "label": "Generalized anxiety disorder"
    },

    "suicidal_ideation_or_self_harm": {
        "domain": "Clinical",
        "label": "Suicidal ideation/self-harm"
    },

    "substance_use_disorder": {
        "domain": "Clinical",
        "label": "Substance use disorder"
    },

    "chronic_pain": {
        "domain": "Clinical",
        "label": "Chronic pain"
    },

    "sleep_disorder": {
        "domain": "Clinical",
        "label": "Sleep disorder"
    },

    "obesity": {
        "domain": "Clinical",
        "label": "Obesity"
    },

    "food_insecurity": {
        "domain": "SDOH",
        "label": "Food insecurity"
    },

    "social_isolation": {
        "domain": "SDOH",
        "label": "Social isolation"
    },

    "unsafe_neighborhood": {
        "domain": "SDOH",
        "label": "Unsafe neighborhood"
    }
}


clinical_outcomes_2visits = [
    outcome
    for outcome, metadata
    in outcome_metadata_2visits.items()
    if metadata["domain"] == "Clinical"
]

sdoh_outcomes_2visits = [
    outcome
    for outcome, metadata
    in outcome_metadata_2visits.items()
    if metadata["domain"] == "SDOH"
]

In [ ]:
# ==================================================
# OUTCOME SANITY CHECKS
# ==================================================

for outcome in clinical_outcomes_2visits:

    analysis_df_2visits[outcome] = (
        analysis_df_2visits[outcome]
        .fillna(0)
        .astype(int)
    )

    assert analysis_df_2visits[
        outcome
    ].isin([0, 1]).all()


for outcome in sdoh_outcomes_2visits:

    valid_values = analysis_df_2visits.loc[
        analysis_df_2visits[outcome].notna(),
        outcome
    ]

    assert valid_values.isin([0, 1]).all(), (
        f"{outcome} contains nonbinary values."
    )

## Supplementary Table S13

Outcome prevalence in the restricted two-visit cohort.


In [ ]:
# ==================================================
# SUPPLEMENTARY TABLE S13:
# OUTCOME PREVALENCE IN THE >=2 VISIT COHORT
# ==================================================

s13_rows = []

for outcome, metadata in outcome_metadata_2visits.items():

    # Outcome-specific denominator:
    # clinical outcomes use everyone;
    # SDOH outcomes use valid respondents only.
    outcome_data = (
        analysis_df_2visits[
            outcome
        ]
        .dropna()
        .astype(int)
    )

    n_total = int(
        outcome_data.shape[0]
    )

    n_with_outcome = int(
        outcome_data.sum()
    )

    percent_with_outcome = (
        100 * n_with_outcome / n_total
        if n_total > 0
        else np.nan
    )

    s13_rows.append({
        "Outcome": metadata["label"],
        "n with outcome": n_with_outcome,
        "n total": n_total,
        "% with outcome": percent_with_outcome
    })


supplementary_table_s13 = pd.DataFrame(
    s13_rows
)


supplementary_table_s13[
    "% with outcome"
] = (
    supplementary_table_s13[
        "% with outcome"
    ]
    .round(2)
)


display(supplementary_table_s13)

In [ ]:
# ==================================================
# SUPPLEMENTARY TABLE S13 VALIDATION
# ==================================================

assert len(
    supplementary_table_s13
) == len(outcome_metadata_2visits)


for outcome, metadata in outcome_metadata_2visits.items():

    outcome_label = metadata["label"]

    expected_n = int(
        analysis_df_2visits[
            outcome
        ].notna().sum()
    )

    expected_cases = int(
        analysis_df_2visits[
            outcome
        ]
        .fillna(0)
        .sum()
    )

    table_row = supplementary_table_s13.loc[
        supplementary_table_s13[
            "Outcome"
        ].eq(outcome_label)
    ].iloc[0]

    assert int(
        table_row["n total"]
    ) == expected_n

    assert int(
        table_row["n with outcome"]
    ) == expected_cases


print(
    "All Supplementary Table S13 "
    "validation checks passed."
)

## Supplementary Table S14

Continuous eight-domain ACE associations in the restricted two-visit cohort.


In [ ]:
# ==================================================
# STANDARDIZE DEMOGRAPHICS FOR S14
# ==================================================

s14_analysis_df = (
    analysis_df_2visits
    .copy()
)


# Keep age buckets consistent
s14_analysis_df["age_group"] = (
    s14_analysis_df["age_group"]
    .replace({
        "<=30": "18-30",
        "≤30": "18-30"
    })
)


valid_age_groups = [
    "18-30",
    "31-45",
    "46-60",
    "61-75",
    "76+"
]


s14_analysis_df.loc[
    ~s14_analysis_df[
        "age_group"
    ].isin(valid_age_groups),
    "age_group"
] = "Missing"


for column in [
    "age_group",
    "gender",
    "race",
    "ethnicity"
]:
    s14_analysis_df[column] = (
        s14_analysis_df[column]
        .astype("object")
        .where(
            s14_analysis_df[
                column
            ].notna(),
            "Missing"
        )
        .astype(str)
    )


# Confirm intended references exist
reference_categories = {
    "age_group": "61-75",
    "gender": "Female",
    "race": "White",
    "ethnicity": "Not Hispanic or Latino"
}


for variable, reference in (
    reference_categories.items()
):
    assert reference in s14_analysis_df[
        variable
    ].unique(), (
        f'Reference category "{reference}" '
        f'is absent from {variable}.'
    )

In [ ]:
# ==================================================
# S14 HELPER FUNCTIONS
# ==================================================

def format_p_value(p_value):

    if pd.isna(p_value) or not np.isfinite(p_value):
        return "NE"

    if p_value < 0.001:
        return "<0.001"

    return (
        f"{p_value:.4f}"
        .rstrip("0")
        .rstrip(".")
    )


def format_or_ci(
    odds_ratio,
    ci_lower,
    ci_upper
):

    if not all(
        np.isfinite(value)
        for value in [
            odds_ratio,
            ci_lower,
            ci_upper
        ]
    ):
        return "NE"

    return (
        f"{odds_ratio:.2f} "
        f"({ci_lower:.2f}–{ci_upper:.2f})"
    )


def fit_two_visit_ace_model(
    df,
    outcome,
    adjusted=False
):
    """
    Logistic regression for the association between
    a one-domain increase in 8-domain ACE burden and
    each binary outcome among participants with >=2 visits.
    """

    required_columns = [
        outcome,
        "ace_burden_score"
    ]

    if adjusted:
        required_columns += [
            "age_group",
            "gender",
            "race",
            "ethnicity"
        ]

    model_df = (
        df[required_columns]
        .dropna()
        .copy()
    )

    model_df[outcome] = (
        model_df[outcome]
        .astype(int)
    )

    model_n = len(model_df)
    cases_n = int(
        model_df[outcome].sum()
    )

    prevalence_pct = (
        100 * cases_n / model_n
        if model_n > 0
        else np.nan
    )

    if (
        model_n == 0
        or cases_n == 0
        or cases_n == model_n
    ):
        return {
            "N": model_n,
            "Cases": cases_n,
            "Prevalence, %": prevalence_pct,
            "OR": np.nan,
            "CI Lower": np.nan,
            "CI Upper": np.nan,
            "p-value raw": np.nan
        }

    if adjusted:

        formula = f"""
            {outcome}
            ~ ace_burden_score
            + C(
                age_group,
                Treatment(reference="61-75")
              )
            + C(
                gender,
                Treatment(reference="Female")
              )
            + C(
                race,
                Treatment(reference="White")
              )
            + C(
                ethnicity,
                Treatment(
                    reference="Not Hispanic or Latino"
                )
              )
        """

    else:

        formula = f"""
            {outcome}
            ~ ace_burden_score
        """

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        result = smf.glm(
            formula=formula,
            data=model_df,
            family=sm.families.Binomial()
        ).fit(
            maxiter=1000
        )

    beta = result.params[
        "ace_burden_score"
    ]

    p_value = result.pvalues[
        "ace_burden_score"
    ]

    confidence_interval = (
        result.conf_int()
        .loc["ace_burden_score"]
    )

    odds_ratio = np.exp(beta)

    ci_lower = np.exp(
        confidence_interval.iloc[0]
    )

    ci_upper = np.exp(
        confidence_interval.iloc[1]
    )

    return {
        "N": model_n,
        "Cases": cases_n,
        "Prevalence, %": prevalence_pct,
        "OR": odds_ratio,
        "CI Lower": ci_lower,
        "CI Upper": ci_upper,
        "p-value raw": p_value
    }

In [ ]:
# ==================================================
# SUPPLEMENTARY TABLE S14:
# CONTINUOUS ACE ASSOCIATIONS IN >=2 VISIT COHORT
# ==================================================

s14_rows = []

for outcome, metadata in (
    outcome_metadata_2visits.items()
):

    for adjusted in [
        False,
        True
    ]:

        model_stats = fit_two_visit_ace_model(
            df=s14_analysis_df,
            outcome=outcome,
            adjusted=adjusted
        )

        s14_rows.append({
            "Outcome Domain": metadata["domain"],
            "Outcome": metadata["label"],
            "Model": (
                "Adjusted"
                if adjusted
                else "Unadjusted"
            ),
            "N": model_stats["N"],
            "Cases": model_stats["Cases"],
            "Prevalence, %": model_stats[
                "Prevalence, %"
            ],
            "OR (95% CI)": format_or_ci(
                model_stats["OR"],
                model_stats["CI Lower"],
                model_stats["CI Upper"]
            ),
            "p-value": format_p_value(
                model_stats["p-value raw"]
            )
        })


supplementary_table_s14 = pd.DataFrame(
    s14_rows
)


supplementary_table_s14[
    "Prevalence, %"
] = (
    supplementary_table_s14[
        "Prevalence, %"
    ]
    .round(1)
)


supplementary_table_s14 = supplementary_table_s14[
    [
        "Outcome Domain",
        "Outcome",
        "Model",
        "N",
        "Cases",
        "Prevalence, %",
        "OR (95% CI)",
        "p-value"
    ]
]


display(supplementary_table_s14)

In [ ]:
# ==================================================
# SUPPLEMENTARY TABLE S14 VALIDATION
# ==================================================

expected_rows = (
    len(outcome_metadata_2visits) * 2
)

assert len(
    supplementary_table_s14
) == expected_rows


for outcome, metadata in (
    outcome_metadata_2visits.items()
):

    outcome_label = metadata["label"]

    expected_n = int(
        s14_analysis_df[
            outcome
        ].notna().sum()
    )

    expected_cases = int(
        s14_analysis_df[
            outcome
        ]
        .fillna(0)
        .sum()
    )

    outcome_rows = supplementary_table_s14.loc[
        supplementary_table_s14[
            "Outcome"
        ].eq(outcome_label)
    ]

    assert len(outcome_rows) == 2

    assert outcome_rows[
        "N"
    ].eq(expected_n).all(), (
        f"{outcome_label}: incorrect N."
    )

    assert outcome_rows[
        "Cases"
    ].eq(expected_cases).all(), (
        f"{outcome_label}: incorrect case count."
    )

    assert set(
        outcome_rows["Model"]
    ) == {
        "Unadjusted",
        "Adjusted"
    }


print(
    "All Supplementary Table S14 "
    "validation checks passed."
)

In [ ]:
# ==================================================
# SAVE OUTPUTS
# ==================================================

supplementary_table_s13.to_csv(
    "outputs/tables/supplementary/supplementary_table_s13_outcome_prevalence_among_participants_with_at_least_2_visits.csv",
    index=False
)

supplementary_table_s14.to_csv(
    "outputs/tables/supplementary/supplementary_table_s14_continuous_8_domain_ace_associations_among_participants_with_at_least_2_visits.csv",
    index=False
)


print(
    "Saved: "
    "outputs/tables/supplementary/supplementary_table_s13_outcome_prevalence_among_participants_with_at_least_2_visits.csv"
)

print(
    "Saved: "
    "outputs/tables/supplementary/supplementary_table_s14_continuous_8_domain_ace_associations_among_participants_with_at_least_2_visits.csv"
)

## Supplementary Figure S3

Comparison of adjusted associations in the main and restricted two-visit cohorts.

**Required inputs:** Supplementary Tables S9 and S14. Run those sections first if their exported CSV files are not present.


In [ ]:
# -----------------------------
# Sensitivity analysis:
# Main cohort vs. participants with ≥2 visits
# -----------------------------

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re

# -----------------------------
# Figure font
# -----------------------------

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [
        "Helvetica",
        "Arial",
        "Liberation Sans",
        "Nimbus Sans",
        "DejaVu Sans",
        "sans-serif"
    ],
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


# -----------------------------
# Load tables
# -----------------------------

main = pd.read_csv(
    "outputs/tables/supplementary/supplementary_table_s9_adjusted_associations_between_continuous_8_domain_ace_burden_and_outcomes.csv"
)

restricted = pd.read_csv(
    "outputs/tables/supplementary/supplementary_table_s14_continuous_8_domain_ace_associations_among_participants_with_at_least_2_visits.csv"
)


# -----------------------------
# Keep adjusted restricted-cohort models
# -----------------------------

restricted = restricted[
    restricted["Model"].astype(str).str.strip().str.lower().eq("adjusted")
].copy()


# -----------------------------
# Standardize outcome names
# -----------------------------

outcome_replacements = {
    "MDD": "Major depressive disorder"
}

main["Outcome"] = main["Outcome"].replace(outcome_replacements)
restricted["Outcome"] = restricted["Outcome"].replace(outcome_replacements)


# -----------------------------
# Function to extract OR and CI
# Handles hyphens, en dashes, and em dashes
# -----------------------------

def extract_or_ci(series):

    extracted = (
        series.astype(str)
        .str.replace("–", "-", regex=False)
        .str.replace("—", "-", regex=False)
        .str.extract(
            r"(?P<or>\d+(?:\.\d+)?)\s*"
            r"\(\s*(?P<ci_low>\d+(?:\.\d+)?)\s*-\s*"
            r"(?P<ci_high>\d+(?:\.\d+)?)\s*\)"
        )
    )

    return extracted.astype(float)


# -----------------------------
# Extract main-cohort estimates
# -----------------------------

main_estimates = extract_or_ci(
    main["Adjusted OR (95% CI)"]
)

main[[
    "main_or",
    "main_ci_low",
    "main_ci_high"
]] = main_estimates.to_numpy()


# -----------------------------
# Extract restricted-cohort estimates
# -----------------------------

restricted_estimates = extract_or_ci(
    restricted["OR (95% CI)"]
)

restricted[[
    "restricted_or",
    "restricted_ci_low",
    "restricted_ci_high"
]] = restricted_estimates.to_numpy()


# -----------------------------
# Select and combine columns
# -----------------------------

main_plot = main[
    [
        "Outcome Domain",
        "Outcome",
        "N",
        "Cases",
        "Prevalence, %",
        "main_or",
        "main_ci_low",
        "main_ci_high"
    ]
].copy()

main_plot = main_plot.rename(columns={
    "Outcome Domain": "outcome_domain",
    "Outcome": "outcome",
    "N": "main_n",
    "Cases": "main_cases",
    "Prevalence, %": "main_prevalence"
})

restricted_plot = restricted[
    [
        "Outcome",
        "N",
        "Cases",
        "Prevalence, %",
        "restricted_or",
        "restricted_ci_low",
        "restricted_ci_high"
    ]
].copy()

restricted_plot = restricted_plot.rename(columns={
    "Outcome": "outcome",
    "N": "restricted_n",
    "Cases": "restricted_cases",
    "Prevalence, %": "restricted_prevalence"
})

compare_df = main_plot.merge(
    restricted_plot,
    on="outcome",
    how="inner",
    validate="one_to_one"
)


# -----------------------------
# Calculate differences
# -----------------------------

compare_df["delta_or_restricted_minus_main"] = (
    compare_df["restricted_or"] -
    compare_df["main_or"]
)

compare_df["absolute_delta_or"] = (
    compare_df["delta_or_restricted_minus_main"].abs()
)


# -----------------------------
# Create formatted estimates
# -----------------------------

compare_df["main_or_ci"] = compare_df.apply(
    lambda row: (
        f'{row["main_or"]:.2f} '
        f'({row["main_ci_low"]:.2f}–{row["main_ci_high"]:.2f})'
    ),
    axis=1
)

compare_df["restricted_or_ci"] = compare_df.apply(
    lambda row: (
        f'{row["restricted_or"]:.2f} '
        f'({row["restricted_ci_low"]:.2f}–'
        f'{row["restricted_ci_high"]:.2f})'
    ),
    axis=1
)


# -----------------------------
# Validate all 12 outcomes
# -----------------------------

assert len(compare_df) == 12, (
    f"Expected 12 matched outcomes, but found {len(compare_df)}."
)

assert compare_df[
    [
        "main_or",
        "main_ci_low",
        "main_ci_high",
        "restricted_or",
        "restricted_ci_low",
        "restricted_ci_high"
    ]
].notna().all().all(), (
    "At least one OR or confidence interval could not be parsed."
)


# -----------------------------
# Display comparison table
# -----------------------------

comparison_display = compare_df.sort_values(
    "main_or",
    ascending=False
)[
    [
        "outcome_domain",
        "outcome",
        "main_or_ci",
        "restricted_or_ci",
        "delta_or_restricted_minus_main",
        "absolute_delta_or"
    ]
]

display(comparison_display)


# -----------------------------
# Order plot from strongest to weakest
# Strongest appears at the top
# -----------------------------

plot_df = compare_df.sort_values(
    "main_or",
    ascending=True
).reset_index(drop=True)


# -----------------------------
# Forest plot
# -----------------------------

fig, ax = plt.subplots(
    figsize=(9, 6.5)
)

y = np.arange(
    len(plot_df)
)

offset = 0.16


# Main cohort
ax.errorbar(
    plot_df["main_or"],
    y + offset,
    xerr=[
        plot_df["main_or"] - plot_df["main_ci_low"],
        plot_df["main_ci_high"] - plot_df["main_or"]
    ],
    fmt="o",
    markersize=5,
    capsize=3,
    linewidth=1.2,
    label="Main cohort"
)


# Restricted cohort
ax.errorbar(
    plot_df["restricted_or"],
    y - offset,
    xerr=[
        plot_df["restricted_or"] - plot_df["restricted_ci_low"],
        plot_df["restricted_ci_high"] - plot_df["restricted_or"]
    ],
    fmt="s",
    markersize=5,
    capsize=3,
    linewidth=1.2,
    label="Participants with ≥2 visits"
)


# Null line
ax.axvline(
    1,
    linestyle="--",
    linewidth=1
)


# Axis labels
ax.set_yticks(
    y
)

ax.set_yticklabels(
    plot_df["outcome"]
)

ax.set_xlabel(
    "Adjusted OR per 1-Point Increase in ACE Burden Score"
)

ax.set_ylabel("")

ax.set_title(
    "Main and Restricted-Cohort Associations Between ACE Burden and Outcomes"
)


# -----------------------------
# Axis limits
# -----------------------------

all_ci_lows = pd.concat([
    plot_df["main_ci_low"],
    plot_df["restricted_ci_low"]
])

all_ci_highs = pd.concat([
    plot_df["main_ci_high"],
    plot_df["restricted_ci_high"]
])

ax.set_xlim(
    min(0.98, all_ci_lows.min() - 0.02),
    all_ci_highs.max() + 0.04
)


# -----------------------------
# Formatting
# -----------------------------

ax.legend(
    loc="lower right",
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()


# ==================================================
# SAVE FIGURE
# ==================================================

fig.savefig(
    "outputs/figures/supplementary/supplementary_figure_s3_main_vs_participants_with_at_least_2_visits_adjusted_associations.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    "outputs/figures/supplementary/supplementary_figure_s3_main_vs_participants_with_at_least_2_visits_adjusted_associations.pdf",
    bbox_inches="tight"
)


# -----------------------------
# Save plotting data
# -----------------------------

compare_df.to_csv(
    "outputs/figure_data/supplementary/supplementary_figure_s3_main_vs_participants_with_at_least_2_visits_plot_data.csv",
    index=False
)

plt.show()

## Completion


In [ ]:
print('Analysis pipeline complete.')
print('Review outputs/ and complete disclosure review before committing results.')
